<a href="https://colab.research.google.com/github/impos0108/AI4WeatherandClimate/blob/main/WeatherBench_Philippines_HeavyRain_DL_with_exp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌧️ Philippines Heavy Rain Prediction using Deep Learning
## WeatherBench1 Low-Resolution Data — Binary Classification Hands-On Workshop

---

**Welcome, PAGASA participants!** 🇵🇭

In this hands-on notebook, we will build a system that **automatically predicts whether heavy rain will occur in the Philippines 6 hours from now**, using large-scale atmospheric data and deep learning.

---

### 🗺️ Big Picture — What are we building?

```
Atmospheric state at time T          →   Will it rain heavily at T+6h?
─────────────────────────────────         ──────────────────────────────
• Geopotential (500 hPa)                  YES 🌧️  (Heavy Rain)
• Specific Humidity (850 hPa)         →   NO  ☀️  (No Heavy Rain)
• Temperature (850 hPa)
• U/V Wind (850 hPa)
```

We will train and compare **four deep learning architectures**:

| Model | Core Idea | Analogy |
|-------|-----------|---------|
| 🔵 **MLP** | Fully connected layers | A calculator with many layers |
| 🟢 **CNN** | Spatial filters on weather maps | Recognising patterns in a photo |
| 🟡 **LSTM** | Memory across time steps | Reading a diary day by day |
| 🔴 **Transformer** | Attention across all time steps | Reading the whole diary at once |

---

### ⚠️ Before you start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run cells **top to bottom** in order — each cell depends on the previous one
3. Read the explanation above each code cell before running it

---

### 📚 Background: What is WeatherBench?
WeatherBench is a public benchmark dataset for data-driven weather forecasting,
built from **ERA5 reanalysis** data produced by ECMWF.
We use the **5.625° resolution** version (~600 km per grid point), which gives a
compact grid that is fast to train on — perfect for a classroom setting.

---
## 📦 Step 1: Install & Import Libraries

### 🤔 Why do we need libraries?
Writing deep learning code from scratch would take months.
Libraries give us **pre-built, optimised tools**:

| Library | What it does |
|---------|-------------|
| **PyTorch** | Build and train neural networks on GPU |
| **xarray** | Handle multi-dimensional weather data (like NetCDF/zarr) |
| **NumPy / Pandas** | Fast numerical computation and data tables |
| **Scikit-learn** | Evaluation metrics and data preprocessing |
| **Matplotlib / Seaborn** | Visualisation |

> 💡 **Tip:** You only need to run the `pip install` cell once per Colab session.

In [ ]:
# Install required packages (only needed in Colab)
!pip install zarr xarray netcdf4 fsspec gcsfs s3fs -q
print("✅ Installation complete!")

#### What happens when we `import` a library?
Python loads the library's code into memory and gives it a short nickname
(`np` for NumPy, `pd` for Pandas, `xr` for xarray, etc.).
After this cell runs, every subsequent cell can use these tools.

**Device setup:** We check whether a GPU is available.
A GPU can perform thousands of mathematical operations *simultaneously*,
making neural network training 10–50× faster than a CPU.

In [ ]:
# === Core Libraries ===
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# === Deep Learning (PyTorch) ===
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn.functional as F

# === Scikit-learn for Metrics & Preprocessing ===
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, balanced_accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

# === Visualization ===
import seaborn as sns
from IPython.display import display

# === Device Setup ===
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
else:
    print("   ⚠️  No GPU detected. Training will be slower. Please enable GPU in Runtime settings.")

# === Random Seed for Reproducibility ===
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print("\n✅ All libraries imported successfully!")

---
## 🌏 Step 2: Load WeatherBench Data

---

### 📌 Why Do We Need a Benchmark Dataset?

In traditional weather forecasting, NWP (Numerical Weather Prediction) models
like ECMWF or GFS are evaluated against observations every day in operations.
But when a new **deep learning** model is proposed, how do we know if it is
actually better?

Without a shared benchmark, every research group uses different:
- Data sources and preprocessing steps
- Train/validation/test splits
- Evaluation metrics

This makes it **impossible to fairly compare** models.

> 💡 Think of it like a standardised exam:
> without a common test, you cannot compare students from different schools.

**WeatherBench** solves this by providing:

| What | Detail |
|------|--------|
| **Fixed dataset** | ERA5 reanalysis, same variables, same resolution |
| **Fixed splits** | Train up to 2016, Val 2017, Test 2018 (WB1) |
| **Fixed metrics** | RMSE, ACC, and task-specific scores |
| **Public leaderboard** | Any team can submit results and compare |

This is why WeatherBench became the **standard benchmark** for data-driven
weather forecasting — the same role that ImageNet played for computer vision.

---

### 🏗️ WeatherBench1 vs WeatherBench2

| | WeatherBench1 (2020) | WeatherBench2 (2023) |
|---|---|---|
| **Paper** | Rasp et al., *arXiv:2002.00469* | Rasp et al., *BAMS 2024* |
| **Data source** | ERA5 (ECMWF) | ERA5 (ECMWF) |
| **Resolution options** | 5.625°, 2.8°, 1.4° | 0.25°, 1.5°, 5.625° |
| **Time range** | 1979–2018 | 1959–2023 |
| **Variables** | 7 atmospheric variables | 20+ variables |
| **Task** | Regression (predict value) | Regression + probabilistic |
| **Baselines** | Persistence, climatology, IFS | IFS HRES, ENS, ML models |
| **Access** | `gs://weatherbench` ⚠️ now closed | `gs://weatherbench2` ✅ public |
| **State-of-the-art** | Simple CNNs | GraphCast, Pangu-Weather, FourCastNet |

> ⚠️ **WeatherBench1** (`gs://weatherbench`) is **no longer publicly accessible**.
> We use **WeatherBench2** in this workshop, which provides the same
> low-resolution ERA5 data in a compatible format.

---

### 🌍 What is ERA5 Reanalysis?

ERA5 is produced by **ECMWF** (European Centre for Medium-Range Weather Forecasts)
by combining numerical model output with real observations from:

```
Observations                  NWP Model
─────────────────             ─────────────────────
• Weather stations            • Physical equations
• Radiosondes (balloons)  +   • Fluid dynamics       →  ERA5 Reanalysis
• Satellites                  • Thermodynamics
• Ships & buoys               • Data assimilation
```

The result is a **physically consistent, gap-free** gridded dataset of the
global atmosphere going back to 1959 — even in regions with no observations.

**ERA5 key facts:**
- Temporal resolution: **hourly** (we use 6-hourly in WeatherBench2)
- Spatial resolution: native **0.25°** (~28 km), downsampled to **5.625°** here
- Vertical levels: 137 model levels (we use **500 hPa** and **850 hPa**)
- Used by: research centres, airlines, energy companies, insurance firms

---

### 📐 The 5.625° Grid — Why Low Resolution?

At 5.625° resolution, one grid cell covers roughly **600 km × 600 km** —
roughly the size of **Luzon island**:

```
 Global grid: 64 × 32 = 2,048 grid points
 Philippines 5×5 region: 25 grid points

 5.625° ≈ 600 km per cell

 25.3°N  ┌─────┬─────┬─────┬─────┬─────┐
         │     │     │     │     │     │
 19.7°N  ├─────┼─────┼─────┼─────┼─────┤
         │     │     │     │     │     │
 14.1°N  ├─────┼─────┼─────┼─────┼─────┤
         │     │     │     │     │     │
  8.4°N  ├─────┼─────┼─────┼─────┼─────┤
         │     │     │     │     │     │
  2.8°N  └─────┴─────┴─────┴─────┴─────┘
       112.5° 118.1° 123.8° 129.4° 135.0°E
```

**Pros of low resolution:**
- ✅ Very small file size → fast to download in Colab
- ✅ Captures large-scale patterns: monsoon flow, typhoon steering, ITCZ
- ✅ Simple models (MLP, CNN) can learn from 25 grid points
- ✅ Training takes minutes, not days

**Cons:**
- ⚠️ Cannot resolve individual thunderstorms (~10 km scale)
- ⚠️ Cannot represent local topography (Cordillera, Sierra Madre)
- ⚠️ One grid cell covers both sea and land — mixed signal

> 💡 **For PAGASA operations:** higher resolution data (1–4 km) from
> WRF or NWCSAF would be needed for actual warning systems.
> This workshop demonstrates the *methodology* using a tractable dataset.

---

### 🔬 Variables We Use

| Variable | Level | Physical Meaning | Why it matters for rain |
|----------|-------|-----------------|------------------------|
| `z` — Geopotential | 500 hPa | Height of the 500 hPa pressure surface | Low z = upper trough → rainfall |
| `q` — Specific Humidity | 850 hPa | Water vapour content in lower atmosphere | High q = moisture available for convection |
| `t` — Temperature | 850 hPa | Low-level temperature | High t + high q = instability → convection |
| `u`, `v` — Wind | 850 hPa | Zonal and meridional wind components | Convergence of wind → uplift → rainfall |
| `tp` — Total Precipitation | Surface | 6-hourly accumulated rainfall | **Our target variable** |

> 💡 **Why 850 hPa?**
> 850 hPa (~1,500 m altitude) is in the **lower troposphere**, just above
> the planetary boundary layer. This is where monsoon moisture transport,
> typhoon inflow, and low-level convergence occur — the direct triggers
> of heavy rainfall in the Philippines.

> ⚠️ **Unit note:** ERA5 `total_precipitation` is stored in **metres (m)**.
> We convert to **mm** by multiplying by 1,000.
> 1 m of water = 1,000 mm = an extreme rainfall event!
> PAGASA warnings use mm/hour or mm/day, so this conversion is essential.


#### 🔧 Configuration Cell
This cell defines all the key parameters in one place.
**You can experiment by changing these values** — that is the point of this workshop!

Key parameters to try:
- `HEAVY_RAIN_THRESHOLD` — lower value → more events labeled as heavy rain
- `USE_PERCENTILE_THRESHOLD` — switch between fixed mm threshold and data-adaptive threshold
- `SEQ_LEN` — how many past time steps LSTM/Transformer look back

In [ ]:
# ============================================================
#  🔧 CONFIGURATION — You can adjust these parameters!
# ============================================================

# Philippines region — exact WB2 64×32 equiangular grid points
# WB2 lat grid = linspace(-87.1875, 87.1875, 32), step = 5.625°
# Using explicit point lists guarantees 5×5 regardless of floating-point
# rounding in xarray .sel(slice(...))
PH_LATS = [2.8125, 8.4375, 14.0625, 19.6875, 25.3125]  # 5 lat points (°N)
PH_LONS = [112.5, 118.125, 123.75, 129.375, 135.0]      # 5 lon points (°E)

# Keep MIN/MAX for map extent (used by make_ph_map)
LAT_MIN, LAT_MAX = PH_LATS[0],  PH_LATS[-1]
LON_MIN, LON_MAX = PH_LONS[0],  PH_LONS[-1]

# ── Heavy Rain Threshold ─────────────────────────────────────
# Method 1 (fixed): set USE_PERCENTILE_THRESHOLD = False
#   → label = 1 if area-max precipitation >= HEAVY_RAIN_THRESHOLD
#   → change HEAVY_RAIN_THRESHOLD to adjust the class ratio
HEAVY_RAIN_THRESHOLD = 10.0   # mm/6h  (try: 5, 10, 20, 50)
# Note: ERA5 raw tp unit is METERS. The load-data cell converts
#       to mm (×1000) before this threshold is applied.

# Method 2 (percentile-based): set USE_PERCENTILE_THRESHOLD = True
#   → threshold = P{PERCENTILE_THRESHOLD} of pixel-max in training period
#   → class ratio is always fixed at (100 - PERCENTILE_THRESHOLD)%
#   → changing HEAVY_RAIN_THRESHOLD has NO effect in this mode
USE_PERCENTILE_THRESHOLD = False  # True = percentile | False = fixed threshold
PERCENTILE_THRESHOLD     = 90    # top-N% as heavy rain (only used when above = True)

# Forecast lead time
LEAD_TIME_HOURS = 6   # 6-hour lead time

# Sequence length for LSTM/Transformer (look-back window)
SEQ_LEN = 8   # 8 steps × 6h = 48-hour look-back

# Training configuration
# Note: tuned for synthetic data (3×3 grid, ~4000 train samples)
#       For real high-res data: increase BATCH_SIZE, LR; decrease DROPOUT
BATCH_SIZE    = 32    # small batch → more gradient updates per epoch
N_EPOCHS      = 80    # more room to learn gradually
LEARNING_RATE = 1e-4  # lower LR → prevents bias-hack at epoch 1
DROPOUT       = 0.4   # slightly lower → easier to learn signal
PATIENCE      = 20    # wait longer before stopping
LR_PATIENCE   = 8     # wait longer before reducing LR
WARMUP_EPOCHS = 5     # gradually ramp up LR for first N epochs

# Train/Val/Test split (years)
TRAIN_YEARS = ('2015', '2016', '2017')
VAL_YEARS   = ('2018',)
TEST_YEARS  = ('2019',)

print("🔧 Configuration:")
print(f"   Region      : Lat [{LAT_MIN}°N – {LAT_MAX}°N], Lon [{LON_MIN}°E – {LON_MAX}°E]")
if USE_PERCENTILE_THRESHOLD:
    print(f"   Threshold   : Percentile-based  P{PERCENTILE_THRESHOLD} of pixel-max (training period)")
    print(f"                 → class ratio fixed at ~{100-PERCENTILE_THRESHOLD}% heavy rain")
    print(f"                 → HEAVY_RAIN_THRESHOLD ({HEAVY_RAIN_THRESHOLD} mm) is IGNORED in this mode")
else:
    print(f"   Threshold   : Fixed  >= {HEAVY_RAIN_THRESHOLD} mm/6h  (pixel-max per timestep)")
print(f"   Lead time   : +{LEAD_TIME_HOURS} hours")
print(f"   Look-back   : {SEQ_LEN} steps ({SEQ_LEN*6}h) for LSTM/Transformer")
print(f"   Train years : {TRAIN_YEARS}")
print(f"   Val year    : {VAL_YEARS}")
print(f"   Test year   : {TEST_YEARS}")


#### 📥 Loading Data from Google Cloud Storage

WeatherBench2 stores ERA5 data as **zarr** format on Google Cloud Storage (GCS).
The bucket is **publicly available** but Colab needs a Google account sign-in.

**Two-step process:**
1. **Step 2a** — Sign in with your Google account (one click, popup appears)
2. **Step 2b** — Data loads automatically after sign-in


In [ ]:
# ============================================================
#  🔐 Step 2a: Google Account Authentication
# ============================================================
# WeatherBench2 data is stored in a PUBLIC Google Cloud bucket,
# but Colab still needs you to sign in to a Google account to
# access GCS (even for public buckets).
#
# ▶ Run this cell → a popup will ask you to sign in with your
#   Google account → click Allow → come back and run Step 2b.
#
# If you don't have a Google account, skip to Step 2c (synthetic data).

try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Google authentication successful!")
    print("   You can now access public GCS buckets.")
except Exception as e:
    print(f"⚠️  Authentication skipped: {e}")
    print("   If you are not on Colab, GCS access may still work anonymously.")


In [ ]:
# ============================================================
#  📥 Step 2b: Load ERA5 Data via WeatherBench2 (GCS)
# ============================================================
# Dataset : WeatherBench2 ERA5  64×32 grid (≈ 5.625°, same as WeatherBench1)
# Path    : gs://weatherbench2/datasets/era5/
#           1959-2022-6h-64x32_equiangular_conservative.zarr
# Access  : Public bucket — requires Google sign-in (Step 2a above)
# Ref     : Rasp et al. (2024), WeatherBench2, BAMS

import gcsfs

WB2_PATH = ('gs://weatherbench2/datasets/era5/'
            '1959-2022-6h-64x32_equiangular_conservative.zarr')

print("📥 Opening WeatherBench2 ERA5 zarr store...")
print(f"   {WB2_PATH}\n")

variables = {}

try:
    # After auth.authenticate_user() above, use default credentials (not anon)
    fs    = gcsfs.GCSFileSystem()          # uses your Google account
    store = fs.get_mapper(WB2_PATH)
    ds    = xr.open_zarr(store, consolidated=True)

    print("   Available variables:", list(ds.data_vars)[:10], "...")
    print("   Dimensions:", dict(ds.dims))

    # Rename lat/lon if needed
    rename = {}
    if 'latitude'  in ds.coords: rename['latitude']  = 'lat'
    if 'longitude' in ds.coords: rename['longitude'] = 'lon'
    if rename:
        ds = ds.rename(rename)

    # Helper: select pressure level if the variable has a level dimension
    def sel_lev(ds, var, lev):
        da = ds[var]
        return da.sel(level=lev, method='nearest') if 'level' in da.dims else da

    # Extract variables (try common WB2 naming conventions)
    print()
    for alias, var_key, lev in [
        (['total_precipitation_6hr', 'total_precipitation', 'tp'], 'tp',   None),
        (['geopotential', 'z'],                                     'z500', 500),
        (['specific_humidity', 'q'],                                'q850', 850),
        (['temperature', 't'],                                      't850', 850),
        (['u_component_of_wind', 'u'],                              'u850', 850),
        (['v_component_of_wind', 'v'],                              'v850', 850),
    ]:
        for name in alias:
            if name in ds:
                da = sel_lev(ds, name, lev) if lev else ds[name]
                variables[var_key] = da
                suffix = f" @ {lev} hPa" if lev else " (surface)"
                print(f"   ✅ {var_key:5s} ← {name}{suffix}")
                break
        else:
            print(f"   ⚠️  {var_key:5s} not found in dataset")

    # ── Unit conversion: ERA5 tp is in METERS → convert to mm ──
    # ERA5 total_precipitation is accumulated in metres of water.
    # Multiply by 1000 to get mm, which is the standard unit for
    # rainfall warnings (e.g. PAGASA uses mm/6h thresholds).
    if 'tp' in variables:
        variables['tp'] = variables['tp'] * 1000.0
        variables['tp'].attrs['units'] = 'mm/6h'
        print("   ✅ tp converted: m → mm  (×1000)")

    if len(variables) >= 5:
        t0 = str(list(variables.values())[0].time.values[0])[:10]
        t1 = str(list(variables.values())[0].time.values[-1])[:10]
        print(f"\n✅ WeatherBench2 data loaded!")
        print(f"   Time range : {t0} → {t1}")
        USE_SYNTHETIC = False
    else:
        raise ValueError(f"Only {len(variables)}/6 variables found: {list(variables.keys())}")

except Exception as e:
    print(f"\n⚠️  GCS load failed: {e}")
    print("\n→ Possible fixes:")
    print("   1. Run Step 2a (Google auth) first, then retry this cell")
    print("   2. Go to Runtime → Disconnect and reconnect, then re-run from Step 2a")
    raise  # stop here — do not continue without real data


---
## 🗺️ Step 3: Extract Philippines Region & Explore Data

### 🤔 Why Exploratory Data Analysis (EDA)?
Before building any model, we need to **understand our data**:
- What does the precipitation distribution look like?
- How often does heavy rain occur? (class balance)
- Which variables are most correlated with heavy rain?
- Are there seasonal patterns we should be aware of?

**Skipping EDA is one of the most common mistakes** in machine learning projects.
A model trained on misunderstood data will give misleading results.

In [ ]:
# ============================================================
#  🗺️ Extract Philippines Region (exact 5×5 grid points)
# ============================================================
# Use explicit point selection instead of slice to guarantee
# exactly 5×5 regardless of floating-point rounding.

print("🗺️ Extracting Philippines 5×5 grid...")

ph_data = {}
for var_name, da in variables.items():
    # Rename latitude/longitude → lat/lon if needed
    rename = {}
    if 'latitude'  in da.dims: rename['latitude']  = 'lat'
    if 'longitude' in da.dims: rename['longitude'] = 'lon'
    if rename:
        da = da.rename(rename)

    # Select exact grid points (method='nearest' handles tiny float offsets)
    ph_data[var_name] = da.sel(lat=PH_LATS, lon=PH_LONS, method='nearest')

# Verify shape
sample = ph_data['tp']
N_LAT  = len(sample.lat)
N_LON  = len(sample.lon)
N_VARS = len(ph_data)

print(f"\n📐 Philippines grid:")
print(f"   Lat ({N_LAT}): {sample.lat.values}")
print(f"   Lon ({N_LON}): {sample.lon.values}")
print(f"   Time steps : {len(sample.time)}")
print(f"   Grid shape : {N_LAT}×{N_LON} = {N_LAT*N_LON} pixels  ✅")
print(f"   Variables  : {list(ph_data.keys())} ({N_VARS} total)")

tp_attrs = ph_data['tp'].attrs
print(f"   tp units   : {tp_attrs.get('units', 'mm/6h (converted)')}")
print(f"   tp sample  : min={float(ph_data['tp'].isel(time=100).values.min()):.3f}, "
      f"max={float(ph_data['tp'].isel(time=100).values.max()):.3f} mm/6h")

assert N_LAT == 5 and N_LON == 5, f"Expected 5×5, got {N_LAT}×{N_LON}!"
print("\n✅ 5×5 grid confirmed!")


#### ⚡ Why load `.values` only once?
xarray DataArrays are **lazy** — they don't load data until you ask for it.
Every call to `.values` triggers a network request to GCS.
By calling `.values` **once** and storing the result as a NumPy array,
all subsequent calculations use fast in-memory operations.

```
ph_data['tp'].values          ← network fetch (slow, ~30s on GCS)
tp_np.mean(axis=(1,2))        ← pure NumPy (instant)
```

#### 🎯 Pixel-MAX threshold
Instead of the spatial mean (which hides extremes),
we use the **maximum precipitation across all pixels** at each timestep.
This captures the most intense event in the region — exactly what
an operational forecaster cares about.

In [ ]:
# ============================================================
#  📊 EDA Common Setup — Run this cell first!
# ============================================================
# Optimization: load .values once, then use numpy for all computations

YEAR_START = TRAIN_YEARS[0]   # '2015'
YEAR_END   = TEST_YEARS[-1]   # '2019'

print(f"Loading tp data... (slicing {YEAR_START}–{YEAR_END} only)")
print("   GCS: ~30s–1min | Synthetic: ~1s")

tp_np = (ph_data['tp']
         .sel(time=slice(YEAR_START, YEAR_END))
         .values)   # load once!
times = (ph_data['tp']
         .sel(time=slice(YEAR_START, YEAR_END))
         .time.values)

print(f"   Done: shape={tp_np.shape}, dtype={tp_np.dtype}")

# All subsequent calculations use fast numpy
tp_area_mean = tp_np.mean(axis=(1, 2))   # spatial mean time series (for visualization)
tp_pixelmax  = tp_np.max(axis=(1, 2))    # per-timestep pixel maximum
tp_vals_pos  = tp_np[tp_np > 0.01].ravel()
tp_spatial   = tp_np.mean(axis=0)        # (lat, lon) mean map

# Threshold — must match the logic in the feature-engineering cell
tr_time_mask = np.array([pd.Timestamp(t).year in [int(y) for y in TRAIN_YEARS]
                          for t in times])

if USE_PERCENTILE_THRESHOLD:
    # Percentile of pixel-max computed on training period only (no leakage)
    eda_thr   = np.percentile(tp_pixelmax[tr_time_mask], PERCENTILE_THRESHOLD)
    thr_label = f"P{PERCENTILE_THRESHOLD} pixel-max = {eda_thr:.1f} mm/6h (training period)"
else:
    # Fixed threshold applied to pixel-max per timestep
    eda_thr   = HEAVY_RAIN_THRESHOLD
    thr_label = f"Fixed {eda_thr:.1f} mm/6h (pixel-max per timestep)"

heavy_mask     = tp_pixelmax > eda_thr
heavy_count    = int(heavy_mask.sum())
no_heavy_count = len(tp_pixelmax) - heavy_count
ratio          = no_heavy_count / heavy_count if heavy_count > 0 else float('inf')
heavy_label    = heavy_mask.astype(float)

# Monthly frequency and rolling mean for plots
tp_timeseries = pd.Series(tp_pixelmax, index=pd.DatetimeIndex(times))
heavy_series  = tp_timeseries > eda_thr
monthly_freq  = heavy_series.groupby(heavy_series.index.month).mean() * 100
tp_rolling    = pd.Series(tp_area_mean,
                           index=pd.DatetimeIndex(times)).rolling(28, center=True).mean()

print(f"\n✅ EDA setup complete! ({YEAR_START}–{YEAR_END})")
print(f"   Threshold     : {thr_label}")
print(f"   Total steps   : {len(tp_pixelmax):,}")
print(f"   Heavy rain    : {heavy_count:,}  ({heavy_count/len(tp_pixelmax)*100:.1f}%)")
print(f"   Class ratio   : {ratio:.1f}:1  (no-rain : heavy-rain)")
print("\n▶ Run the plot cells below one by one.")


#### 📊 Plot 1 — Precipitation Distribution
Precipitation is **zero-inflated** (most of the time it doesn't rain)
and **right-skewed** (a few extreme events dominate).
The log scale on the y-axis makes the rare heavy-rain tail visible.

In [ ]:
# 📊 Plot 1/6 — Precipitation Distribution — Precipitation Distribution (log scale)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(tp_vals_pos, bins=60, color='steelblue', edgecolor='white', alpha=0.8, log=True)
ax.axvline(eda_thr, color='red', linewidth=2,
           label=f'Heavy rain threshold: {eda_thr:.1f} mm/6h')
ax.set_xlabel('Total Precipitation (mm/6h)')
ax.set_ylabel('Count (log scale)')
ax.set_title('Precipitation Distribution')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


#### 📊 Plot 2 — Monthly Heavy Rain Frequency
The Philippines has a pronounced **monsoon cycle**.
Heavy rain should concentrate in the wet season (Jun–Nov),
driven by the Southwest Monsoon and typhoons.
If your synthetic or real data shows this pattern, the data is realistic.

In [ ]:
# 📊 Plot 2/6 — Monthly Heavy Rain Frequency — Monthly Heavy Rain Frequency
months = ['Jan','Feb','Mar','Apr','May','Jun',
          'Jul','Aug','Sep','Oct','Nov','Dec']
colors = ['steelblue' if (m < 6 or m > 11) else 'tomato'
          for m in range(1, 13)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 13), monthly_freq.values, color=colors, edgecolor='white')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months, fontsize=9)
ax.set_ylabel('Heavy Rain Frequency (%)')
ax.set_title('Monthly Heavy Rain Frequency\n🔴 Wet Season (Jun–Nov)   🔵 Dry Season')
ax.axvspan(5.5, 11.5, alpha=0.1, color='blue', label='Wet Season')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


#### 📊 Plot 3 — Class Imbalance ⚠️ Important concept!
Heavy rain events are **rare** — this is called **class imbalance**.

**Why does this matter for machine learning?**
Imagine a model that always predicts "No Rain".
On a dataset where rain occurs 10% of the time, this model achieves
**90% accuracy** while being completely useless for warnings!

This is why we use metrics like **CSI, F1, AUC** instead of accuracy,
and why we apply **class weighting** during training to penalise
missed heavy rain events more heavily than false alarms.

In [ ]:
# 📊 Plot 3/6 — Class Imbalance — Class Imbalance
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    [no_heavy_count, heavy_count],
    labels=[f'No Heavy Rain\n({no_heavy_count:,})', f'Heavy Rain\n({heavy_count:,})'],
    colors=['#5B9BD5', '#FF6B6B'],
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10}
)
ax.set_title(f'Class Distribution\n{thr_label}\nRatio {ratio:.1f}:1  ⚠️ Imbalanced!')
plt.tight_layout()
plt.show()
print(f"💡 Heavy rain = {heavy_count/len(tp_pixelmax)*100:.1f}% of all timesteps.")
print("   → Use CSI, F1, AUC instead of raw Accuracy!")


#### 📊 Plot 4 — Precipitation Time Series
The rolling mean smooths out noise and reveals the **seasonal cycle**.
Spikes above the threshold are the heavy rain events our model must learn to predict.

In [ ]:
# 📊 Plot 4/6 — Precipitation Time Series — Precipitation Time Series
fig, ax = plt.subplots(figsize=(12, 4))
tp_timeseries.plot(ax=ax, alpha=0.3, color='steelblue',
                   linewidth=0.5, label='6-hourly pixel-max')
tp_rolling.plot(ax=ax, color='navy', linewidth=1.5, label='7-day rolling mean')
ax.axhline(eda_thr, color='red', linestyle='--', linewidth=1.5,
           label=f'Threshold ({eda_thr:.1f} mm/6h)')
ax.set_ylabel('Pixel-max Precipitation (mm/6h)')
ax.set_title('Philippines Pixel-Max Precipitation Time Series')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


#### 📊 Plot 5 — Spatial Map
Even at coarse 5.625° resolution, we can see **spatial patterns** in rainfall.
In reality, the eastern coast of the Philippines receives more rain
due to the trade winds and typhoon tracks — does your map reflect this?

In [ ]:
# ============================================================
#  🗺️ Map Drawing Setup (Cartopy)
#  Run once — defines helper functions used by all map plots
# ============================================================
!pip install cartopy -q

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

def make_ph_map(ax=None, figsize=(7, 6), title=''):
    """
    Create a Cartopy GeoAxes centred on the Philippines grid.
    Returns (fig, ax) ready for data overlays.
    """
    proj = ccrs.PlateCarree()
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize,
                               subplot_kw={'projection': proj})
    else:
        fig = ax.get_figure()

    ax.set_extent([LON_MIN - 4, LON_MAX + 4,
                   LAT_MIN - 3, LAT_MAX + 3], crs=proj)

    ax.add_feature(cfeature.OCEAN,     facecolor='#d6eaf8', zorder=0)
    ax.add_feature(cfeature.LAND,      facecolor='#f0e6d3', zorder=1)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7,        zorder=2)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.4,
                   linestyle='--', edgecolor='grey',         zorder=2)
    ax.add_feature(cfeature.RIVERS,    linewidth=0.3,
                   edgecolor='steelblue', alpha=0.5,         zorder=2)

    gl = ax.gridlines(draw_labels=True, linewidth=0.4,
                      color='grey', alpha=0.5, linestyle='--')
    gl.top_labels   = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 7}
    gl.ylabel_style = {'size': 7}

    if title:
        ax.set_title(title, fontsize=10, fontweight='bold')
    return fig, ax


def plot_grid_on_map(data_2d, lon_vals, lat_vals, ax,
                     cmap='Blues', vmin=None, vmax=None,
                     alpha=0.75):
    """
    Overlay a (n_lat x n_lon) array on a Cartopy map.

    Uses imshow with extent + transform=PlateCarree().
    This approach is immune to shape mismatches — imshow only
    needs the bounding extent [lon_min, lon_max, lat_min, lat_max],
    not per-cell edge coordinates.

    Works for any grid size (4x5, 5x5, etc.)
    """
    proj  = ccrs.PlateCarree()
    dlon  = (lon_vals[1] - lon_vals[0]) / 2 if len(lon_vals) > 1 else 2.8125
    dlat  = (lat_vals[1] - lat_vals[0]) / 2 if len(lat_vals) > 1 else 2.8125
    extent = [lon_vals[0]  - dlon, lon_vals[-1] + dlon,
              lat_vals[0]  - dlat, lat_vals[-1] + dlat]

    im = ax.imshow(data_2d,
                   origin='lower',          # lat increases upward
                   extent=extent,
                   transform=proj,
                   cmap=cmap,
                   vmin=vmin, vmax=vmax,
                   alpha=alpha,
                   interpolation='nearest', # keep blocky grid cells
                   zorder=3)
    return im


print("✅ Cartopy map helpers ready  (imshow-based, shape-agnostic)")
print("   make_ph_map()      — Philippines base map with coastlines")
print("   plot_grid_on_map() — overlay grid data (any n_lat × n_lon)")


In [ ]:
# 📊 Plot 5/6 — Spatial Mean Precipitation Map (on Philippines map)
# tp_spatial already computed in eda-setup (no reload needed)
import cartopy.crs as ccrs

# Use coordinates from ph_data — guaranteed to match tp_spatial shape
lat_vals = ph_data['tp'].sel(time=slice(YEAR_START, YEAR_END)).lat.values  # (n_lat,)
lon_vals = ph_data['tp'].sel(time=slice(YEAR_START, YEAR_END)).lon.values  # (n_lon,)
print(f"tp_spatial shape: {tp_spatial.shape}  | lat: {len(lat_vals)}  lon: {len(lon_vals)}")

fig, ax = make_ph_map(figsize=(8, 7),
                      title=f'Mean Precipitation — Philippines Grid ({len(lat_vals)}×{len(lon_vals)}, 5.625°)')

im = plot_grid_on_map(tp_spatial, lon_vals, lat_vals, ax, cmap='Blues', vmin=0)
plt.colorbar(im, ax=ax, label='Mean Precip (mm/6h)', shrink=0.7, pad=0.02)

# Annotate each grid point — iterate using tp_spatial shape, not lat/lon length
for r in range(tp_spatial.shape[0]):
    for c in range(tp_spatial.shape[1]):
        lat, lon = lat_vals[r], lon_vals[c]
        val = tp_spatial[r, c]
        ax.plot(lon, lat, 'o', color='navy', ms=5,
                transform=ccrs.PlateCarree(), zorder=5)
        ax.text(lon + 0.3, lat + 0.3, f'{val:.1f}',
                fontsize=6.5, color='navy',
                transform=ccrs.PlateCarree(), zorder=6)

plt.tight_layout()
plt.show()

print(f"\n✅ EDA complete!")
print(f"   Heavy rain events : {heavy_count:,} / {len(tp_pixelmax):,} ({heavy_count/len(tp_pixelmax)*100:.1f}%)")
print(f"   Class ratio       : {ratio:.1f}:1")


#### 📊 Plot 6 — Predictor Correlation (run last — slowest!)
Pearson correlation tells us whether a variable has a **linear relationship**
with heavy rain occurrence. Variables with high absolute correlation
are likely to be useful predictors.

> ⚠️ **Limitation:** Deep learning can capture *non-linear* relationships
> that simple correlation misses. A low correlation doesn't mean
> a variable is useless — it might still contribute to the model.

In [ ]:
# 📊 Plot 6/6 — Predictor Correlation with Heavy Rain — Predictor Correlation with Heavy Rain
# Slice each variable to the same time range as heavy_label (YEAR_START–YEAR_END)
cors = {}
for var_name, da in ph_data.items():
    if var_name == 'tp':
        continue
    var_area = (da.sel(time=slice(YEAR_START, YEAR_END))
                  .values
                  .mean(axis=(1, 2)))   # shape (T,) — matches heavy_label
    cors[var_name] = np.corrcoef(var_area, heavy_label)[0, 1]

cor_df = pd.Series(cors).sort_values(key=abs, ascending=False)
colors_cor = ['tomato' if c < 0 else 'steelblue' for c in cor_df.values]

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(cor_df.index, cor_df.values, color=colors_cor, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Heavy Rain')
ax.set_title('Predictor Correlation with Heavy Rain')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()
print("💡 Positive (blue): higher value → more likely heavy rain")
print("   Negative (red) : lower value  → more likely heavy rain")


---
## 🛠️ Step 4: Feature Engineering & Dataset Preparation

### 🤔 What is feature engineering?
Raw data is rarely in the right format for a neural network.
We need to:
1. **Define the prediction task** — what is X (input) and y (output)?
2. **Align X and y in time** — predict y at t+6h using X at time t
3. **Normalise** — scale all inputs to a similar range
4. **Split** — separate data into train / validation / test sets
5. **Create data loaders** — feed data to the model in batches

### ⏰ Temporal alignment
```
Time:    t=0    t=1    t=2    t=3    ...
X:    [vars]  [vars]  [vars]  [vars]   ← atmospheric state (input)
y:            [rain?] [rain?] [rain?]  ← heavy rain 6h later (target)
```
X at time t predicts y at time t+1 (one 6-hour step ahead).

### 📅 Why split by year, not randomly?
Weather data is **temporally correlated** — today's atmosphere is similar
to yesterday's. If we split randomly, the model sees future data during
training (data leakage), which artificially inflates performance.
Splitting by year gives a **fair evaluation** on truly unseen future data.

```
2015 ──── 2016 ──── 2017   │  2018   │  2019
      TRAINING              │   VAL   │   TEST
```

#### 🏷️ Label Creation
We create a **binary label** (0 or 1) for each 6-hour timestep:
- `1` = Heavy Rain (pixel-max precipitation exceeds threshold)
- `0` = No Heavy Rain

Using **pixel-max** (maximum across all grid points) rather than
the spatial mean ensures we capture intense localised events,
such as a typhoon hitting just one corner of the domain.

In [ ]:
# ============================================================
#  🛠️ Feature Engineering
# ============================================================

print("🛠️ Preparing features and labels...")

# --- Stack predictor variables → (T, n_vars, lat, lon) ---
PREDICTOR_VARS = ['z500', 'q850', 't850', 'u850', 'v850']
N_PRED_VARS    = len(PREDICTOR_VARS)

X_stack = np.stack(
    [ph_data[v].sel(time=slice(YEAR_START, YEAR_END)).values
     for v in PREDICTOR_VARS],
    axis=1
)  # (T, n_vars, n_lat, n_lon)

# --- Heavy rain labels ---
# tp_np already loaded in eda-setup (no reload)
tp_3d = tp_np   # (T, lat, lon)

# Training-period mask for threshold calculation (prevents data leakage)
tr_time_mask = np.array([pd.Timestamp(t).year in [int(y) for y in TRAIN_YEARS]
                          for t in times])

# Per-timestep pixel maximum — captures the most extreme event in the region
tp_pixelmax = tp_3d.max(axis=(1, 2))   # (T,)

if USE_PERCENTILE_THRESHOLD:
    # Percentile threshold: top-(100-N)% of pixel-max in training period
    # Pros: class ratio is always fixed regardless of absolute rainfall values
    # Cons: HEAVY_RAIN_THRESHOLD (mm) is ignored
    thr = np.percentile(tp_pixelmax[tr_time_mask], PERCENTILE_THRESHOLD)
    print(f"\n📏 Threshold mode : Percentile (P{PERCENTILE_THRESHOLD}, training period)")
    print(f"   tp_pixelmax range : {tp_pixelmax.min():.1f} – {tp_pixelmax.max():.1f} mm/6h")
    print(f"   P{PERCENTILE_THRESHOLD} threshold     : {thr:.2f} mm/6h")
    print(f"   Note: HEAVY_RAIN_THRESHOLD ({HEAVY_RAIN_THRESHOLD} mm) is NOT used in this mode")
else:
    # Fixed threshold applied to pixel-max per timestep
    # → change HEAVY_RAIN_THRESHOLD in CONFIG to adjust class ratio
    thr = HEAVY_RAIN_THRESHOLD
    print(f"\n📏 Threshold mode : Fixed  >= {thr:.1f} mm/6h  (pixel-max per timestep)")

y_raw = (tp_pixelmax > thr).astype(np.float32)

# Visualize threshold on distribution
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(tp_pixelmax, bins=60, color='steelblue', edgecolor='white', alpha=0.8, log=True)
ax.axvline(thr, color='red', linewidth=2, label=f'Threshold = {thr:.1f} mm/6h')
ax.set_xlabel('Pixel-max Precipitation per timestep (mm/6h)')
ax.set_ylabel('Count (log)')
ax.set_title('Distribution of Per-Timestep Pixel-Max Precipitation')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

n_time = X_stack.shape[0]
print(f"\n   Input shape  : (T={n_time}, n_vars={N_PRED_VARS}, lat={N_LAT}, lon={N_LON})")

# --- Align X(t) → y(t+1) for 6-hour lead time ---
X_all     = X_stack[:-1]
y_all     = y_raw[1:]
times_all = (ph_data['tp']
             .sel(time=slice(YEAR_START, YEAR_END))
             .time.values[:-1])

print(f"   X_all shape  : {X_all.shape}")
print(f"   y_all shape  : {y_all.shape}")
print(f"   Heavy rain   : {y_all.mean()*100:.1f}%  (target: 5–15%)")

# --- Train / Val / Test split ---
def get_year_mask(times, years):
    years_int = [int(y) for y in years]
    return np.array([pd.Timestamp(t).year in years_int for t in times])

tr_mask   = get_year_mask(times_all, TRAIN_YEARS)
val_mask  = get_year_mask(times_all, VAL_YEARS)
test_mask = get_year_mask(times_all, TEST_YEARS)

X_tr,  y_tr  = X_all[tr_mask],   y_all[tr_mask]
X_val, y_val = X_all[val_mask],  y_all[val_mask]
X_te,  y_te  = X_all[test_mask], y_all[test_mask]

print(f"\n📂 Dataset splits:")
print(f"   Train  ({', '.join(TRAIN_YEARS)}): {len(X_tr):,} samples | Heavy rain: {y_tr.mean()*100:.1f}%")
print(f"   Val    ({', '.join(VAL_YEARS)}) : {len(X_val):,} samples | Heavy rain: {y_val.mean()*100:.1f}%")
print(f"   Test   ({', '.join(TEST_YEARS)}) : {len(X_te):,} samples | Heavy rain: {y_te.mean()*100:.1f}%")


#### 📏 Normalisation — Why is this essential?
Neural networks learn by computing **gradients** (slopes) of the loss function.
If input features have very different scales (e.g., geopotential ~5000 m²/s²
vs humidity ~0.015 kg/kg), the gradients become unbalanced and training
is slow or unstable.

**StandardScaler** transforms each feature to have:
- Mean = 0
- Standard deviation = 1

> ⚠️ **Critical rule:** We compute mean and std **only on the training set**,
> then apply the same transformation to validation and test.
> Using val/test statistics would be **data leakage** — the model would
> implicitly "know" something about the future during training.

```python
scaler.fit(X_train)        # learn mean/std from training data only
scaler.transform(X_train)  # apply to train
scaler.transform(X_val)    # apply same transform to val
scaler.transform(X_test)   # apply same transform to test
```

In [ ]:
# ============================================================
#  📏 Normalization (using training set statistics only)
# ============================================================
# IMPORTANT: We compute mean/std ONLY on training data
# to avoid data leakage into validation/test sets

print("📏 Normalizing features (StandardScaler on training set)...")

# Flatten spatial dims for scaling: (T, n_vars×n_lat×n_lon)
X_tr_flat  = X_tr.reshape(len(X_tr), -1).astype(np.float32)
X_val_flat = X_val.reshape(len(X_val), -1).astype(np.float32)
X_te_flat  = X_te.reshape(len(X_te), -1).astype(np.float32)

scaler = StandardScaler()
X_tr_scaled  = scaler.fit_transform(X_tr_flat)   # Fit ONLY on training!
X_val_scaled = scaler.transform(X_val_flat)       # Transform val/test with training stats
X_te_scaled  = scaler.transform(X_te_flat)

# Reshape back to (T, n_vars, n_lat, n_lon) for CNN
X_tr_cnn  = X_tr_scaled.reshape(-1, N_PRED_VARS, N_LAT, N_LON)
X_val_cnn = X_val_scaled.reshape(-1, N_PRED_VARS, N_LAT, N_LON)
X_te_cnn  = X_te_scaled.reshape(-1, N_PRED_VARS, N_LAT, N_LON)

# For MLP: flat (T, n_vars×n_lat×n_lon)
INPUT_SIZE_MLP = X_tr_scaled.shape[1]

print(f"\n   Input size for MLP: {INPUT_SIZE_MLP} features")
print(f"   Input shape for CNN: ({N_PRED_VARS}, {N_LAT}, {N_LON})")
print(f"   ✅ Normalization done — mean ~0, std ~1 on training set")

# --- Class weights for handling imbalance ---
# Give more weight to the rare heavy rain class
pos_count  = y_tr.sum()
neg_count  = len(y_tr) - pos_count
raw_ratio  = neg_count / (pos_count + 1e-8)

# Use sqrt(ratio) instead of raw ratio as pos_weight.
# Raw ratio (~9) is too aggressive: model learns "always predict positive"
# in epoch 1, causing val loss to spike and early stopping to fire immediately.
# sqrt(ratio) (~3) gently upweights positives without destabilising training.
POS_WEIGHT = torch.tensor([np.sqrt(raw_ratio)], dtype=torch.float32).to(DEVICE)

print(f"\n⚖️  Class imbalance handling:")
print(f"   neg/pos ratio    : {raw_ratio:.1f}:1")
print(f"   pos_weight (sqrt): {POS_WEIGHT.item():.2f}  (sqrt of ratio — gentler than raw ratio)")

#### 🗃️ PyTorch Dataset and DataLoader

**Dataset** — wraps our NumPy arrays so PyTorch can access individual samples
**DataLoader** — creates **mini-batches** and optionally shuffles the data

**Why mini-batches?**
Training on one sample at a time is slow (no parallelism).
Training on the full dataset at once requires too much GPU memory.
Mini-batches (32–256 samples) balance speed and memory.

**Three dataset types for our four models:**

```
FlatDataset    → MLP    : (batch, n_features)          flat vector
SpatialDataset → CNN    : (batch, channels, H, W)      2D weather map
SequenceDataset→ LSTM   : (batch, seq_len, n_features) time sequence
               → Transformer
```

**SequenceDataset** creates a sliding window:
```
Window 1: [t=0, t=1, ..., t=7] → predict t=8
Window 2: [t=1, t=2, ..., t=8] → predict t=9
...
```

In [ ]:
# ============================================================
#  🗃️ PyTorch Dataset Classes
# ============================================================

class FlatDataset(Dataset):
    """For MLP: flat feature vectors."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class SpatialDataset(Dataset):
    """For CNN: 2D spatial grids (channels, height, width)."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (T, C, H, W)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class SequenceDataset(Dataset):
    """
    For LSTM/Transformer: time sequences.
    Returns (seq_len, n_features) for each sample.
    """
    def __init__(self, X_flat, y, seq_len):
        self.seq_len = seq_len
        # Build valid indices (only after seq_len time steps)
        self.X = torch.tensor(X_flat, dtype=torch.float32)  # (T, features)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X) - self.seq_len
    def __getitem__(self, idx):
        # Return window of seq_len steps ending at idx+seq_len
        x_seq = self.X[idx : idx + self.seq_len]  # (seq_len, features)
        y_val = self.y[idx + self.seq_len]          # label at last step
        return x_seq, y_val


# Build datasets and DataLoaders
def make_loaders(X_flat, X_cnn, y, split='train', seq_len=SEQ_LEN, batch_size=BATCH_SIZE):
    """Create DataLoaders for all model types."""
    shuffle = (split == 'train')

    dl_mlp = DataLoader(
        FlatDataset(X_flat, y),
        batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True
    )
    dl_cnn = DataLoader(
        SpatialDataset(X_cnn, y),
        batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True
    )
    dl_seq = DataLoader(
        SequenceDataset(X_flat, y, seq_len),
        batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True
    )
    return {'mlp': dl_mlp, 'cnn': dl_cnn, 'seq': dl_seq}


loaders_tr  = make_loaders(X_tr_scaled,  X_tr_cnn,  y_tr,  'train')
loaders_val = make_loaders(X_val_scaled, X_val_cnn, y_val, 'val')
loaders_te  = make_loaders(X_te_scaled,  X_te_cnn,  y_te,  'test')

print("✅ DataLoaders created for all model types!")
print(f"   MLP  — batch shape: {next(iter(loaders_tr['mlp']))[0].shape}")
print(f"   CNN  — batch shape: {next(iter(loaders_tr['cnn']))[0].shape}")
print(f"   Seq  — batch shape: {next(iter(loaders_tr['seq']))[0].shape}  (LSTM/Transformer)")

---
## 🧠 Step 5: Define Deep Learning Models

### 🤔 What is a neural network?
A neural network is a mathematical function that maps inputs to outputs
through a series of **layers**. Each layer applies a linear transformation
followed by a non-linear **activation function**.

**ReLU** (Rectified Linear Unit): `f(x) = max(0, x)`
This simple non-linearity allows deep networks to learn complex,
non-linear patterns that simpler models cannot capture.

**Training** adjusts the weights in each layer to minimise the loss
using **backpropagation** + **gradient descent**.

> ▶ Run the cell below to see illustrated diagrams of all four architectures,
> then continue to the model definition code.


In [ ]:
# ============================================================
#  🧠 Deep Learning Architecture Diagrams
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def draw_box(ax, x, y, w, h, label, color, fontsize=8, text_color='white', alpha=0.88):
    rect = mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle="round,pad=0.025",
        linewidth=1.2, edgecolor='white',
        facecolor=color, alpha=alpha, zorder=3
    )
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center',
            fontsize=fontsize, color=text_color,
            fontweight='bold', zorder=4, linespacing=1.3)

def arrow(ax, x1, y1, x2, y2, color='#bbbbbb'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.4), zorder=5)

def lay(n_boxes, top=0.92, bottom=0.10, box_h=0.07):
    """Evenly space n_boxes between top and bottom."""
    gap = (top - bottom - n_boxes * box_h) / (n_boxes - 1) if n_boxes > 1 else 0
    ys = [top - box_h/2 - i*(box_h + gap) for i in range(n_boxes)]
    return ys, box_h

# ── Figure ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 9))
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#1a1a2e')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

# ── 🔵 MLP (6 layers) ────────────────────────────────────────
ax = axes[0]
ax.set_title('🔵 MLP\nMulti-Layer Perceptron',
             color='white', fontsize=11, fontweight='bold', pad=8)

labels = [
    'Input\n125 features\n(5 vars × 5×5)',
    'Dense(125→64)\n+ BatchNorm + ReLU + Dropout',
    'Dense(64→32)\n+ BatchNorm + ReLU + Dropout',
    'Dense(32→1)',
    'Sigmoid',
    'P(Heavy Rain)',
]
colors = ['#2d6a9f','#4472C4','#4472C4','#4472C4','#2d4a8f','#1a3a6f']
ys, bh = lay(6, top=0.90, bottom=0.14)
for i,(y,lbl,col) in enumerate(zip(ys, labels, colors)):
    w = 0.72 if i < 3 else 0.55 if i < 5 else 0.62
    draw_box(ax, 0.5, y, w, bh, lbl, col, fontsize=7.5)
    if i < len(ys)-1:
        arrow(ax, 0.5, y-bh/2, 0.5, ys[i+1]+bh/2)
ax.text(0.5, 0.05, 'All inputs → all neurons\n(fully connected)',
        ha='center', va='center', fontsize=7, color='#999999', style='italic')

# ── 🟢 CNN (7 layers) ────────────────────────────────────────
ax = axes[1]
ax.set_title('🟢 CNN\nConvolutional Neural Network',
             color='white', fontsize=11, fontweight='bold', pad=8)

labels = [
    'Input Map  5×5×5\n(lat × lon × vars)',
    'Conv2d 3×3 → 32 maps\n+ BatchNorm + ReLU',
    'Conv2d 3×3 → 64 maps\n+ BatchNorm + ReLU',
    'Conv2d 3×3 → 128 maps\n+ BatchNorm + ReLU',
    'Global Avg Pool\n→ 128 values',
    'Dense → 1  →  Sigmoid',
    'P(Heavy Rain)',
]
colors = ['#2d6a4f','#70AD47','#5d9e38','#4a8d29','#3a7a1a','#2a6a0a','#1a5a00']
ys, bh = lay(7, top=0.90, bottom=0.14, box_h=0.055)   # ← 박스 높이 줄임
for i,(y,lbl,col) in enumerate(zip(ys, labels, colors)):
    draw_box(ax, 0.5, y, 0.76, bh, lbl, col, fontsize=7.5)
    if i < len(ys)-1:
        arrow(ax, 0.5, y-bh/2, 0.5, ys[i+1]+bh/2)
ax.text(0.5, 0.05, 'Detects spatial patterns in weather maps',
        ha='center', va='center', fontsize=7, color='#999999', style='italic')

# ── 🟡 LSTM ──────────────────────────────────────────────────
ax = axes[2]
ax.set_title('🟡 LSTM\nLong Short-Term Memory',
             color='white', fontsize=11, fontweight='bold', pad=8)

# 6개 박스, 유효 폭 0.86 (양쪽 margin 0.07)
# box_w=0.09, gap=(0.86 - 6*0.09)/5 = 0.064 → step=0.154, start=0.07+0.09/2=0.115
n_steps = 6
BOX_W  = 0.09
STEP   = 0.154
START  = 0.115
xs = [START + i * STEP for i in range(n_steps)]
step_labels = ['t-7','t-6','t-5','…','t-1','t-0']
step_colors = ['#5a4f00','#6b5c00','#7c6900','#6b5c00','#9e8600','#FFC000']
cell_h = 0.07   # ← 박스 높이 줄임

# Row 1: time step boxes  y=0.84
# Row 2: LSTM cells       y=0.68
T_Y, C_Y = 0.84, 0.68
for i,(x,lbl,col) in enumerate(zip(xs, step_labels, step_colors)):
    draw_box(ax, x, T_Y, BOX_W, cell_h, lbl, col, fontsize=7.5)
    draw_box(ax, x, C_Y, BOX_W, cell_h, 'LSTM\ncell', '#cc9900', fontsize=6.5)
    arrow(ax, x, T_Y - cell_h/2, x, C_Y + cell_h/2)
    if i < n_steps - 1:
        half = BOX_W / 2 - 0.005
        ax.annotate('', xy=(xs[i+1] - half, C_Y),
                    xytext=(x + half, C_Y),
                    arrowprops=dict(arrowstyle='->', color='#FFC000', lw=1.4), zorder=5)

# 하단 출력 박스 — 중앙(cx=0.5) 배치로 오른쪽 잘림 방지
lx  = xs[-1]
cx  = 0.5
steps_out = [
    (cx, 0.52, 0.32, cell_h, 'Dense → 1',    '#aa7700'),
    (cx, 0.38, 0.32, cell_h, 'Sigmoid',       '#886600'),
    (cx, 0.22, 0.36, cell_h, 'P(Heavy Rain)', '#664400'),
]
arrow(ax, lx, C_Y - cell_h/2, cx, steps_out[0][1] + cell_h/2)
prev_y = steps_out[0][1] - cell_h/2
for i,(x,y,w,h,lbl,col) in enumerate(steps_out):
    draw_box(ax, x, y, w, h, lbl, col, fontsize=8)
    if i > 0:
        arrow(ax, x, prev_y, x, y + h/2)
    prev_y = y - h/2

ax.text(0.5, 0.09, 'Sequential: processes one\ntime step at a time',
        ha='center', va='center', fontsize=7, color='#999999', style='italic')

# ── 🔴 Transformer ────────────────────────────────────────────
ax = axes[3]
ax.set_title('🔴 Transformer\n(Self-Attention)',
             color='white', fontsize=11, fontweight='bold', pad=8)

# 5개 박스, box_w=0.12, step=0.185, start=0.13
n_steps = 5
T_BOX_W = 0.12
xs_t = [0.13 + i * 0.185 for i in range(n_steps)]
for x,lbl in zip(xs_t, ['t-7','t-5','t-3','t-1','t-0']):
    draw_box(ax, x, 0.88, T_BOX_W, 0.07, lbl, '#991111', fontsize=8)

# Attention lines
for i,xi in enumerate(xs_t):
    for j,xj in enumerate(xs_t):
        if i != j:
            ax.plot([xi, xj], [0.843, 0.785],
                    color='#FF6666', alpha=0.12 + 0.06*abs(i-j),
                    linewidth=0.8, zorder=2)

# Main blocks
main_labels = [
    '⬛ Self-Attention\n(all pairs simultaneously)',
    'Feed-Forward  ×2 layers',
    'Mean Pooling over sequence',
    'Dense → 1  →  Sigmoid',
    'P(Heavy Rain)',
]
main_colors = ['#cc2222','#aa1111','#881111','#661111','#440000']
ys_main, bh_main = lay(5, top=0.74, bottom=0.14, box_h=0.07)

for i,(y,lbl,col) in enumerate(zip(ys_main, main_labels, main_colors)):
    draw_box(ax, 0.5, y, 0.88, bh_main, lbl, col, fontsize=8)
    if i == 0:
        arrow(ax, 0.5, 0.785, 0.5, y + bh_main/2)
    if i < len(ys_main) - 1:
        if i == 0:
            mid_y = (ys_main[0] - bh_main/2 + ys_main[1] + bh_main/2) / 2
            ax.text(0.5, mid_y, '+ Positional Encoding',
                    ha='center', va='center', fontsize=7.5, color='#ff9999', zorder=6)
        arrow(ax, 0.5, y - bh_main/2, 0.5, ys_main[i+1] + bh_main/2)

ax.text(0.5, 0.05, 'Parallel: sees all time steps\nat once via attention',
        ha='center', va='center', fontsize=7, color='#999999', style='italic')

# ── Save ──────────────────────────────────────────────────────
plt.subplots_adjust(left=0.01, right=0.99, top=0.87, bottom=0.01, wspace=0.06)
plt.savefig('model_architectures.png', dpi=130, bbox_inches='tight',
            pad_inches=0.15, facecolor=fig.get_facecolor())
plt.show()
print("MLP: 6 | CNN: 7 | LSTM: seq→dense | Transformer: attn→ff→pool→dense")


In [ ]:
# ============================================================
#  🔵 Model 1: MLP (Multi-Layer Perceptron)
# ============================================================
# The simplest neural network. Takes all features as a flat vector.
# Architecture: Input → Dense → ReLU → Dropout → Dense → Output

class MLPModel(nn.Module):
    """
    Multi-Layer Perceptron for binary classification.

    Input: flat feature vector (n_vars × n_lat × n_lon)
    Output: logit for heavy rain probability
    """
    def __init__(self, input_size, hidden_sizes=[64, 32], dropout=0.5):
        super().__init__()

        layers = []
        prev_size = input_size

        for hidden_size in hidden_sizes:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.BatchNorm1d(hidden_size),  # Normalize activations
                nn.ReLU(),
                nn.Dropout(dropout)           # Prevent overfitting
            ])
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, 1))  # Output: single logit
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)  # (batch,)


# ============================================================
#  🟢 Model 2: CNN (Convolutional Neural Network)
# ============================================================
# Uses 2D convolutions to capture SPATIAL patterns.
# Like image recognition, but for weather maps!

class CNNModel(nn.Module):
    """
    2D CNN for spatial weather pattern recognition.

    Input: (batch, n_vars, n_lat, n_lon) — like a multi-channel image
    Output: logit for heavy rain probability
    """
    def __init__(self, in_channels, n_lat, n_lon, dropout=0.5):
        super().__init__()

        # Convolutional feature extractor
        self.conv_layers = nn.Sequential(
            # Conv block 1
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # Conv block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # Conv block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )

        # Global Average Pooling — collapse spatial dimensions
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.conv_layers(x)    # Extract spatial features
        x = self.global_pool(x)    # (batch, 128, 1, 1)
        x = self.classifier(x)     # (batch, 1)
        return x.squeeze(-1)       # (batch,)


# ============================================================
#  🟡 Model 3: LSTM (Long Short-Term Memory)
# ============================================================
# Recurrent neural network that captures TEMPORAL sequences.
# Remembers past weather to predict future conditions.

class LSTMModel(nn.Module):
    """
    LSTM for temporal weather sequence modeling.

    Input: (batch, seq_len, n_features) — sequence of weather states
    Output: logit for heavy rain probability
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.5):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,      # (batch, seq, features)
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False    # Unidirectional: only past → future
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (batch, seq_len, features)
        out, (h_n, c_n) = self.lstm(x)   # Process sequence
        last = out[:, -1, :]              # Take output at last time step
        return self.classifier(last).squeeze(-1)


# ============================================================
#  🔴 Model 4: Transformer
# ============================================================
# Uses self-attention to capture dependencies across time.
# Currently the most powerful architecture in NLP and increasingly in weather!

class PositionalEncoding(nn.Module):
    """Add position information to sequence embeddings."""
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Compute positional encodings
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             -(np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)  # Even dims: sine
        pe[:, 1::2] = torch.cos(position * div_term)  # Odd dims: cosine
        self.register_buffer('pe', pe.unsqueeze(0))    # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TransformerModel(nn.Module):
    """
    Transformer encoder for temporal weather sequence classification.

    Input: (batch, seq_len, n_features)
    Output: logit for heavy rain probability
    """
    def __init__(self, input_size, d_model=32, nhead=4, num_layers=2,
                 dim_ff=64, dropout=0.2, seq_len=SEQ_LEN):
        super().__init__()

        # Input projection: map features to model dimension
        self.input_proj = nn.Linear(input_size, d_model)

        # Positional encoding
        self.pos_enc = PositionalEncoding(d_model, max_len=seq_len+1, dropout=dropout)

        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head (use mean pooling over sequence)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        # x: (batch, seq_len, features)
        x = self.input_proj(x)     # Project to d_model
        x = self.pos_enc(x)        # Add position info
        x = self.transformer(x)    # Self-attention across time
        x = x.mean(dim=1)          # Mean pooling over sequence
        return self.classifier(x).squeeze(-1)


# Instantiate models
models = {
    'MLP':         MLPModel(INPUT_SIZE_MLP, dropout=DROPOUT),
    'CNN':         CNNModel(N_PRED_VARS, N_LAT, N_LON, dropout=DROPOUT),
    'LSTM':        LSTMModel(INPUT_SIZE_MLP, dropout=DROPOUT),
    'Transformer': TransformerModel(INPUT_SIZE_MLP, seq_len=SEQ_LEN),
}

print("🧠 Model parameter counts:")
for name, model in models.items():
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   {name:12s}: {n_params:>8,} parameters")

---
## 🏋️ Step 6: Training the Models

### 🤔 How does a neural network learn?

Training is an **iterative optimisation** process:

```
1. Forward pass  : feed a batch of X through the model → get predictions ŷ
2. Compute loss  : measure how wrong ŷ is compared to true y
3. Backward pass : compute gradient of loss w.r.t. every weight (backprop)
4. Update weights: move weights slightly in the direction that reduces loss
5. Repeat for next batch → next epoch → until convergence
```

### 📉 Loss Function: Binary Cross-Entropy (BCE)
For binary classification, we use BCE:

```
Loss = -[ y · log(ŷ) + (1-y) · log(1-ŷ) ]
```

- If y=1 (heavy rain) and ŷ≈1 (correct): loss ≈ 0
- If y=1 (heavy rain) and ŷ≈0 (missed!): loss → ∞

**pos_weight** multiplies the loss for positive (heavy rain) cases,
compensating for class imbalance. If the ratio is 9:1, pos_weight=9
means **missing one heavy rain event costs as much as 9 false alarms**.

### ⚙️ Optimizer: Adam
Adam (Adaptive Moment Estimation) is the most popular optimizer
for deep learning. It adapts the learning rate for each weight
individually, combining the benefits of momentum and RMSProp.

### 📅 Early Stopping
We monitor validation loss after each epoch.
If it hasn't improved for `PATIENCE` epochs, training stops.
This prevents **overfitting** — the model memorising the training
data at the expense of generalisation to new data.

*(See the diagram generated by the cell below.)*

### 📐 Learning Rate Scheduler
`ReduceLROnPlateau` automatically halves the learning rate
when validation loss stops improving, allowing finer convergence.

In [ ]:
# ============================================================
#  📉 Early Stopping — Concept Diagram
#  Run this cell to see how early stopping works
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(0)
epochs = np.arange(1, 51)

# Simulated train loss: steadily decreasing
train_loss = 1.2 * np.exp(-0.07 * epochs) + 0.08 + np.random.normal(0, 0.005, len(epochs))

# Simulated val loss: decreases then rises (overfitting)
val_loss = (1.3 * np.exp(-0.09 * epochs)        # decreasing phase
            + 0.25 * (1 - np.exp(-0.05 * (epochs - 20))) * (epochs > 20)  # rising phase
            + 0.12
            + np.random.normal(0, 0.008, len(epochs)))

best_epoch = int(np.argmin(val_loss)) + 1   # epoch with lowest val loss

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(epochs, train_loss, color='steelblue', linewidth=2.5, label='Train Loss')
ax.plot(epochs, val_loss,   color='tomato',    linewidth=2.5, label='Validation Loss')

# Shade overfitting region
ax.axvspan(best_epoch, epochs[-1], alpha=0.08, color='red', label='Overfitting region')

# Mark best epoch
ax.axvline(best_epoch, color='green', linestyle='--', linewidth=2)
label_text = f'Best model saved (epoch {best_epoch})'
ax.annotate(label_text,
            xy=(best_epoch, val_loss[best_epoch - 1]),
            xytext=(best_epoch + 5, val_loss[best_epoch - 1] + 0.13),
            fontsize=10, color='green',
            arrowprops=dict(arrowstyle='->', color='green', lw=2.5))

# Mark patience window
patience = 15
ax.annotate('', xy=(best_epoch + patience, val_loss[best_epoch + patience - 1]),
            xytext=(best_epoch, val_loss[best_epoch + patience - 1]),
            arrowprops=dict(arrowstyle='<->', color='purple', lw=1.5))
ax.text(best_epoch + patience / 2, val_loss[best_epoch + patience - 1] - 0.13,
        f'PATIENCE={patience}\ntraining stops here',
        ha='center', fontsize=9, color='purple')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Early Stopping — How It Works', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print(f"  • Train loss keeps DECREASING throughout — model memorises training data")
print(f"  • Val loss DECREASES then RISES from epoch {best_epoch} — overfitting begins")
print(f"  • We save the model at epoch {best_epoch} (lowest val loss)")
print(f"  • Training stops at epoch {best_epoch + patience} (after PATIENCE={patience} epochs with no improvement)")


In [ ]:
# ============================================================
#  🏋️ Training & Evaluation Functions
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, device):
    """Train model for one epoch. Returns average loss."""
    model.train()
    total_loss = 0.0
    n_batches = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()          # Reset gradients
        logits = model(X_batch)        # Forward pass
        loss = criterion(logits, y_batch)  # Compute loss
        loss.backward()               # Backward pass (compute gradients)

        # Gradient clipping — prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()              # Update weights
        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate model on a DataLoader. Returns loss, predictions, labels."""
    model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []
    n_batches = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        probs = torch.sigmoid(logits)   # Convert logits to probabilities

        total_loss += loss.item()
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())
        n_batches += 1

    return total_loss / n_batches, np.array(all_probs), np.array(all_labels)


def train_model(model_name, model, loader_type, n_epochs=N_EPOCHS,
                lr=LEARNING_RATE, patience=PATIENCE, device=DEVICE):
    """
    Full training loop with early stopping.

    Args:
        model_name: Name for logging
        model: PyTorch model
        loader_type: 'mlp', 'cnn', or 'seq' (for LSTM/Transformer)
        n_epochs: Maximum epochs
        lr: Learning rate
        patience: Early stopping patience

    Returns:
        history: dict with train/val loss per epoch
        best_model_state: best model weights
    """
    model = model.to(device)

    # Loss function: BCEWithLogits handles numerical stability
    # pos_weight upweights the rare heavy rain class
    criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)

    # Adam optimizer with weight decay (L2 regularization)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # Warmup + ReduceLROnPlateau scheduler
    # Warmup: for the first WARMUP_EPOCHS, scale LR from lr/10 → lr
    # This prevents the model from making large, destabilising updates early on
    def warmup_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / WARMUP_EPOCHS  # 0.2, 0.4, 0.6, 0.8, 1.0
        return 1.0

    warmup_scheduler = optim.lr_scheduler.LambdaLR(optimizer, warmup_lambda)
    plateau_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=LR_PATIENCE
    )

    tr_loader  = loaders_tr[loader_type]
    val_loader = loaders_val[loader_type]

    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    print(f"\n{'='*60}")
    print(f"  Training {model_name}")
    print(f"{'='*60}")
    print(f"  {'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val AUC':>8} | {'LR':>8}")
    print(f"  {'-'*50}")

    for epoch in range(1, n_epochs + 1):
        # Training
        tr_loss = train_one_epoch(model, tr_loader, optimizer, criterion, device)

        # Validation
        val_loss, val_probs, val_labels = evaluate(model, val_loader, criterion, device)

        # AUC score
        try:
            val_auc = roc_auc_score(val_labels, val_probs)
        except:
            val_auc = 0.5

        warmup_scheduler.step()
        if epoch >= WARMUP_EPOCHS:
            plateau_scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)

        best_mark = '⭐' if val_loss < best_val_loss else '  '
        print(f"  {epoch:>5} | {tr_loss:>10.4f} | {val_loss:>10.4f} | {val_auc:>8.4f} | {current_lr:.2e} {best_mark}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n  ⏹️  Early stopping at epoch {epoch} (best val loss: {best_val_loss:.4f})")
                break

    print(f"\n  ✅ {model_name} training complete! Best val loss: {best_val_loss:.4f}")
    return history, best_state


print("✅ Training functions defined!")

In [ ]:
# ============================================================
#  🚀 Train All Four Models
# ============================================================
# Each model takes 2-5 minutes on GPU. Be patient! ☕

all_histories = {}
all_best_states = {}

# Mapping model → loader type
LOADER_MAP = {
    'MLP':         'mlp',
    'CNN':         'cnn',
    'LSTM':        'seq',
    'Transformer': 'seq',
}

for model_name, model in models.items():
    loader_type = LOADER_MAP[model_name]
    history, best_state = train_model(
        model_name, model, loader_type,
        n_epochs=N_EPOCHS, lr=LEARNING_RATE
    )
    all_histories[model_name] = history
    all_best_states[model_name] = best_state

print("\n" + "="*60)
print("  🎉 All models trained!")
print("="*60)

---
## 📈 Step 7: Evaluate & Compare Models

### 📏 Why accuracy is not enough

With 10% heavy rain events, a model that **always predicts "No Rain"**
achieves 90% accuracy — but is completely useless operationally!

We use **weather-specific metrics** instead:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **POD** (Probability of Detection) | TP / (TP + FN) | Of all actual heavy rain events, what fraction did we catch? |
| **FAR** (False Alarm Rate) | FP / (FP + TN) | Of all no-rain events, what fraction did we falsely alarm? |
| **CSI** (Critical Success Index) | TP / (TP + FP + FN) | Overall skill, penalises both misses and false alarms |
| **HSS** (Heidke Skill Score) | Skill over random chance | > 0 means better than random; 1.0 = perfect |
| **AUC-ROC** | Area under ROC curve | Overall discriminative ability; 0.5 = random, 1.0 = perfect |
| **F1** | 2·Precision·Recall / (P+R) | Harmonic mean of precision and recall |

> 💡 **For warning systems:** High POD is critical (don't miss disasters),
> but very high FAR erodes public trust. CSI balances both.

### 🔢 Confusion Matrix
```
                  Predicted
                  No Rain  |  Heavy Rain
Actual  No Rain  |   TN    |     FP      ← False Alarm
        Heavy    |   FN    |     TP      ← Hit
                      ↑
                   Miss (most dangerous in operational context!)
```

### 🎚️ Decision Threshold
The model outputs a **probability** (0–1), not a binary prediction.
We choose a threshold (e.g., 0.5) to convert probability to YES/NO.
- **Lower threshold** → more warnings issued → higher POD, higher FAR
- **Higher threshold** → fewer warnings issued → lower POD, lower FAR

We find the **optimal threshold** using the validation set (not test set!).

In [ ]:
# ============================================================
#  📊 Test Set Evaluation
# ============================================================

def compute_metrics(y_true, y_prob, threshold=0.5):
    """Compute comprehensive evaluation metrics."""
    y_pred = (y_prob >= threshold).astype(int)
    y_true = y_true.astype(int)

    # Confusion matrix elements
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

    # Standard metrics
    acc  = (tp + tn) / (tp + tn + fp + fn)
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)    # = Probability of Detection (POD)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    far  = fp / (fp + tn + 1e-8)    # False Alarm Rate

    # Weather-specific scores
    csi  = tp / (tp + fp + fn + 1e-8)  # Critical Success Index (Threat Score)

    # Heidke Skill Score
    n = tp + tn + fp + fn
    expected = ((tp + fp) * (tp + fn) + (tn + fn) * (tn + fp)) / (n * n)
    hss  = (acc - expected) / (1 - expected + 1e-8)

    # AUC
    try:
        auc = roc_auc_score(y_true, y_prob)
    except:
        auc = 0.5

    return {
        'Accuracy': acc, 'Precision': prec, 'Recall (POD)': rec,
        'F1': f1, 'FAR': far, 'CSI': csi, 'HSS': hss,
        'AUC-ROC': auc,
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn
    }


# Evaluate each model on the test set
criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
test_results = {}

print("🧪 Evaluating models on test set...\n")

for model_name, model in models.items():
    # Load best weights
    model.load_state_dict(all_best_states[model_name])
    model = model.to(DEVICE)

    loader_type = LOADER_MAP[model_name]
    te_loader = loaders_te[loader_type]

    _, probs, labels = evaluate(model, te_loader, criterion, DEVICE)

    # Find optimal threshold using validation set
    val_loader = loaders_val[loader_type]
    _, val_probs, val_labels = evaluate(model, val_loader, criterion, DEVICE)
    thresholds = np.arange(0.1, 0.9, 0.05)
    best_thr = max(thresholds,
                   key=lambda t: f1_score(val_labels, (val_probs>=t).astype(int), zero_division=0))

    metrics = compute_metrics(labels, probs, threshold=best_thr)
    metrics['Threshold'] = best_thr
    metrics['probs'] = probs
    metrics['labels'] = labels

    test_results[model_name] = metrics
    print(f"  {model_name:12s} | AUC={metrics['AUC-ROC']:.3f} | F1={metrics['F1']:.3f} | "
          f"CSI={metrics['CSI']:.3f} | HSS={metrics['HSS']:.3f} | thr={best_thr:.2f}")

print("\n✅ Evaluation complete!")

In [ ]:
# ============================================================
#  📋 Results Summary Table
# ============================================================

metric_cols = ['Accuracy', 'Precision', 'Recall (POD)', 'F1', 'FAR', 'CSI', 'HSS', 'AUC-ROC']

df_results = pd.DataFrame({
    name: {m: test_results[name][m] for m in metric_cols}
    for name in test_results
}).T

# Highlight: higher is better except FAR (lower is better)
def color_metric(s, col):
    """Color-code metric values."""
    if col == 'FAR':
        best = s.min()
        return ['background-color: #90EE90' if v == best else '' for v in s]
    else:
        best = s.max()
        return ['background-color: #90EE90' if v == best else '' for v in s]

print("\n📋 Test Set Performance Comparison")
print("="*80)
print(df_results.round(3).to_string())
print("\n  🟢 = Best value per metric")
print("  FAR (False Alarm Rate): lower is better. All others: higher is better.")

display(df_results.style
    .format("{:.3f}")
    .apply(lambda s: color_metric(s, s.name), axis=0)
    .set_caption("Test Set Metrics — Philippines Heavy Rain Prediction")
    .set_properties(**{'text-align': 'center'}))

In [ ]:
# ============================================================
#  📊 Comprehensive Comparison Visualization
# ============================================================

MODEL_COLORS = {
    'MLP':         '#4472C4',
    'CNN':         '#70AD47',
    'LSTM':        '#FFC000',
    'Transformer': '#FF4444',
}

fig = plt.figure(figsize=(20, 18))
fig.suptitle('🌧️ Philippines Heavy Rain Prediction — Model Comparison\n(WeatherBench1 5.625° | +6h Lead Time)',
             fontsize=14, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# --- 1. Training curves ---
ax1 = fig.add_subplot(gs[0, :2])
for model_name, hist in all_histories.items():
    color = MODEL_COLORS[model_name]
    epochs = range(1, len(hist['train_loss']) + 1)
    ax1.plot(epochs, hist['train_loss'], '--', color=color, alpha=0.5, linewidth=1)
    ax1.plot(epochs, hist['val_loss'], '-', color=color, linewidth=2,
             label=f"{model_name} (val)")
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss')
ax1.set_title('Training & Validation Loss Curves\n(solid=val, dashed=train)')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# --- 2. Metric bar chart ---
ax2 = fig.add_subplot(gs[0, 2])
key_metrics = ['AUC-ROC', 'F1', 'CSI', 'HSS']
x = np.arange(len(key_metrics))
width = 0.2
for i, (model_name, _) in enumerate(test_results.items()):
    vals = [test_results[model_name][m] for m in key_metrics]
    ax2.bar(x + i*width, vals, width, label=model_name,
            color=MODEL_COLORS[model_name], edgecolor='white')
ax2.set_xticks(x + width*1.5)
ax2.set_xticklabels(key_metrics, fontsize=9)
ax2.set_ylabel('Score')
ax2.set_title('Key Metrics Comparison')
ax2.legend(fontsize=8)
ax2.set_ylim(0, 1)
ax2.grid(axis='y', alpha=0.3)

# --- 3. ROC Curves ---
ax3 = fig.add_subplot(gs[1, 0])
for model_name, res in test_results.items():
    fpr, tpr, _ = roc_curve(res['labels'], res['probs'])
    auc = res['AUC-ROC']
    ax3.plot(fpr, tpr, color=MODEL_COLORS[model_name], linewidth=2,
             label=f"{model_name} (AUC={auc:.3f})")
ax3.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax3.set_xlabel('False Positive Rate')
ax3.set_ylabel('True Positive Rate (POD)')
ax3.set_title('ROC Curves')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3)

# --- 4. Precision-Recall Curves ---
ax4 = fig.add_subplot(gs[1, 1])
for model_name, res in test_results.items():
    prec_c, rec_c, _ = precision_recall_curve(res['labels'], res['probs'])
    ap = average_precision_score(res['labels'], res['probs'])
    ax4.plot(rec_c, prec_c, color=MODEL_COLORS[model_name], linewidth=2,
             label=f"{model_name} (AP={ap:.3f})")
base = test_results[list(test_results.keys())[0]]['labels'].mean()
ax4.axhline(base, color='gray', linestyle='--', linewidth=1, label=f'Baseline ({base:.2f})')
ax4.set_xlabel('Recall')
ax4.set_ylabel('Precision')
ax4.set_title('Precision-Recall Curves')
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

# --- 5. Confusion matrices ---
for i, (model_name, res) in enumerate(test_results.items()):
    ax = fig.add_subplot(gs[1, 2] if i == 0 else gs[2, i-1] if i < 4 else None)
    if i == 0:
        ax = fig.add_subplot(gs[1, 2])
    elif i == 1:
        ax = fig.add_subplot(gs[2, 0])
    elif i == 2:
        ax = fig.add_subplot(gs[2, 1])
    elif i == 3:
        ax = fig.add_subplot(gs[2, 2])

    cm = np.array([[res['TN'], res['FP']], [res['FN'], res['TP']]])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm_pct, annot=True, fmt='.1f', ax=ax, cmap='Blues',
                xticklabels=['No Rain', 'Heavy Rain'],
                yticklabels=['No Rain', 'Heavy Rain'],
                cbar=False)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{model_name}\nConfusion Matrix (%)', fontsize=10,
                 color=MODEL_COLORS[model_name], fontweight='bold')

plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n📊 Comparison plot saved!")

In [ ]:
# ============================================================
#  📈 Validation AUC Curves During Training
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Progress', fontsize=13, fontweight='bold')

for model_name, hist in all_histories.items():
    color = MODEL_COLORS[model_name]
    epochs = range(1, len(hist['val_auc']) + 1)
    ax1.plot(epochs, hist['val_auc'], color=color, linewidth=2, marker='o',
             markersize=3, label=model_name)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Validation AUC-ROC')
ax1.set_title('Validation AUC During Training')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim(0.4, 1.0)

# Spider/Radar chart
from matplotlib.patches import FancyArrowPatch
metrics_radar = ['AUC-ROC', 'F1', 'CSI', 'HSS', 'Precision', 'Recall (POD)']
N = len(metrics_radar)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

ax2 = fig.add_subplot(1, 2, 2, polar=True)
ax2.set_title('Skill Radar Chart\n(Test Set)', fontsize=11, pad=15)

for model_name, res in test_results.items():
    vals = [res[m] for m in metrics_radar]
    vals += vals[:1]
    ax2.plot(angles, vals, color=MODEL_COLORS[model_name], linewidth=2, label=model_name)
    ax2.fill(angles, vals, color=MODEL_COLORS[model_name], alpha=0.1)

ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(metrics_radar, fontsize=9)
ax2.set_ylim(0, 1)
ax2.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax2.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=7)
ax2.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_progress.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
#  🎯 Rank Models and Provide Interpretation
# ============================================================

print("="*65)
print("  🏆 MODEL RANKING (by AUC-ROC on Test Set)")
print("="*65)

ranking = sorted(test_results.items(), key=lambda x: x[1]['AUC-ROC'], reverse=True)
medals = ['🥇', '🥈', '🥉', '🏅']

for rank, (name, res) in enumerate(ranking):
    medal = medals[rank] if rank < len(medals) else '  '
    print(f"\n  {medal} #{rank+1}: {name}")
    print(f"      AUC-ROC : {res['AUC-ROC']:.3f}")
    print(f"      F1 Score: {res['F1']:.3f}")
    print(f"      CSI     : {res['CSI']:.3f}  ← Weather forecasting key metric")
    print(f"      HSS     : {res['HSS']:.3f}  ← Skill over random chance")
    print(f"      POD     : {res['Recall (POD)']:.3f}  ← Probability of Detection")
    print(f"      FAR     : {res['FAR']:.3f}  ← False Alarm Rate (lower=better)")

print("\n" + "="*65)
print("\n📚 INTERPRETATION GUIDE:")
print()
print("  🔵 MLP  — Simplest but often surprisingly competitive.")
print("            Works well when spatial patterns are less important.")
print()
print("  🟢 CNN  — Captures spatial weather patterns (like cyclone structure).")
print("            Advantage: sees relative positions of features.")
print("            Limitation: small grid (5.625°) limits spatial advantage.")
print()
print("  🟡 LSTM — Captures temporal sequences (evolution of weather).")
print("            Good at: monsoon onset, typhoon intensification patterns.")
print("            Limitation: sequential computation is slow.")
print()
print("  🔴 Transformer — State-of-the-art attention mechanism.")
print("            Can attend to any part of the time sequence directly.")
print("            Best with longer sequences and more data.")
print()
print("⚠️  KEY CAVEAT: With synthetic data, rankings may differ from")
print("   real WeatherBench data. Use this notebook as a template and")
print("   run with real data for production use!")

---
## 🧪 Step 9: Hands-On Exercises

Now it is your turn to experiment!
Each exercise below changes one part of the pipeline so you can see
**what effect it has on model performance**.

> 💡 **Scientific method:** Change **one thing at a time**,
> record the results, and form a hypothesis about why it changed.

In [ ]:
# ============================================================
#  🧪 EXERCISE 1: Change the Rain Threshold
# ============================================================
# Question: How does the threshold affect the class ratio?
# PAGASA warning levels (approximate mm/6h equivalents):
#   - Moderate rain : ~  7.5 mm/hr = 45 mm/6h
#   - Heavy rain    : ~ 15.0 mm/hr = 90 mm/6h
#   - Intense rain  : ~ 30.0 mm/hr = 180 mm/6h
#
# TODO: Change EXERCISE_THRESHOLD and re-run this cell!
EXERCISE_THRESHOLD = 10.0   # ← Try: 5, 20, 50, 100 mm/6h

# ── Use the same approach as feature-engineering ──────────────
# 1. Sliced to YEAR_START~YEAR_END (same period as training)
# 2. Pixel-MAX per timestep (not area mean)
# tp_np is already loaded in eda-setup (no reload needed)
ex_pixelmax = tp_np.max(axis=(1, 2))   # (T,) pixel-max time series

y_exercise  = (ex_pixelmax >= EXERCISE_THRESHOLD).astype(np.float32)
pos_rate    = y_exercise.mean() * 100

print(f"📊 Fixed threshold = {EXERCISE_THRESHOLD} mm/6h  (pixel-max per timestep)")
print(f"   Time period     : {YEAR_START}–{YEAR_END}  ({len(y_exercise):,} timesteps)")
print(f"   Heavy rain freq : {pos_rate:.1f}%  ({int(y_exercise.sum())} events)")
print(f"   No-rain freq    : {100-pos_rate:.1f}%  ({int((1-y_exercise).sum())} events)")
print(f"   Class ratio     : {(1-y_exercise).sum()/max(y_exercise.sum(),1):.1f}:1  (no-rain : heavy)")

# Compare with current model threshold
if USE_PERCENTILE_THRESHOLD:
    tr_mask_ex = np.array([pd.Timestamp(t).year in [int(y) for y in TRAIN_YEARS]
                            for t in times])
    current_thr = np.percentile(ex_pixelmax[tr_mask_ex], PERCENTILE_THRESHOLD)
    print(f"\n   Current model   : P{PERCENTILE_THRESHOLD} = {current_thr:.1f} mm/6h  "
          f"({100-PERCENTILE_THRESHOLD:.0f}% heavy rain)")
else:
    print(f"\n   Current model   : fixed {HEAVY_RAIN_THRESHOLD:.1f} mm/6h  "
          f"({(ex_pixelmax >= HEAVY_RAIN_THRESHOLD).mean()*100:.1f}% heavy rain)")

if pos_rate < 1:
    print("\n   ⚠️  Very rare — model will struggle (too few positive samples to learn from)")
elif pos_rate > 40:
    print("\n   ⚠️  Very common — this is no longer a 'heavy rain' threshold!")
else:
    print("\n   ✅ Reasonable class ratio for learning.")

print("\n💡 Key insight:")
print("   Higher threshold → rarer events → harder to predict → lower F1/CSI")
print("   But operationally, catching extreme events is MORE important!")
print("   → This is why CSI and POD matter more than overall accuracy.")


In [ ]:
# ============================================================
#  🧪 EXERCISE 2: Feature Importance with Permutation Test
# ============================================================
# Which variable is MOST important for prediction?
# Method: Randomly shuffle one variable → measure performance drop

# Select the best model from our comparison
best_model_name = max(test_results, key=lambda k: test_results[k]['AUC-ROC'])
print(f"🔍 Permutation importance test using best model: {best_model_name}")

best_model = models[best_model_name].to(DEVICE)
best_model.load_state_dict(all_best_states[best_model_name])
best_model.eval()

# Baseline performance
loader_type = LOADER_MAP[best_model_name]
_, base_probs, base_labels = evaluate(best_model, loaders_te[loader_type], criterion, DEVICE)
base_auc = roc_auc_score(base_labels, base_probs)
print(f"   Baseline AUC: {base_auc:.4f}")

# Permutation test (only works cleanly for MLP with flat input)
print("\n   Permuting each variable on test set (MLP analysis):")
print(f"   {'Variable':>10} | {'Permuted AUC':>12} | {'AUC Drop':>10} | Importance")
print(f"   {'-'*55}")

mlp_model = models['MLP'].to(DEVICE)
mlp_model.load_state_dict(all_best_states['MLP'])
mlp_model.eval()

X_te_perm_base = torch.tensor(X_te_scaled, dtype=torch.float32)
n_feats_per_var = INPUT_SIZE_MLP // N_PRED_VARS

importance_scores = {}
for i, var_name in enumerate(PREDICTOR_VARS):
    X_perm = X_te_perm_base.clone()
    # Shuffle features belonging to this variable
    start = i * n_feats_per_var
    end   = (i + 1) * n_feats_per_var
    perm_idx = torch.randperm(len(X_perm))
    X_perm[:, start:end] = X_perm[perm_idx, start:end]

    # Evaluate with permuted variable
    with torch.no_grad():
        logits = mlp_model(X_perm.to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()

    perm_auc = roc_auc_score(y_te[:len(probs)], probs)
    drop = base_auc - perm_auc
    importance_scores[var_name] = drop

    bar = '█' * int(abs(drop) * 200)
    print(f"   {var_name:>10} | {perm_auc:>12.4f} | {drop:>+10.4f} | {bar}")

most_important = max(importance_scores, key=importance_scores.get)
print(f"\n   🏆 Most important variable: {most_important}")
print(f"   💡 Removing {most_important} causes the largest AUC drop!")

In [ ]:
# ============================================================
#  🧪 EXERCISE 3: Modify the MLP Architecture
# ============================================================
# Try adding more layers, changing hidden size, or dropout!

# TODO: Modify these parameters and see how performance changes:
EXERCISE_HIDDEN = [512, 256, 128, 64]  # ← Try more/fewer layers!
EXERCISE_DROPOUT = 0.4                  # ← Try 0.1 to 0.6

print("🔧 Custom MLP Architecture Experiment")
print(f"   Hidden layers: {EXERCISE_HIDDEN}")
print(f"   Dropout: {EXERCISE_DROPOUT}")

custom_mlp = MLPModel(INPUT_SIZE_MLP, hidden_sizes=EXERCISE_HIDDEN, dropout=EXERCISE_DROPOUT)
n_params = sum(p.numel() for p in custom_mlp.parameters() if p.requires_grad)
print(f"   Parameters: {n_params:,}")

print("\n   Training custom MLP (5 epochs for quick test)...")
hist_custom, state_custom = train_model(
    'Custom MLP', custom_mlp, 'mlp',
    n_epochs=10, lr=LEARNING_RATE, patience=5
)

custom_mlp.load_state_dict(state_custom)
custom_mlp = custom_mlp.to(DEVICE)
_, c_probs, c_labels = evaluate(custom_mlp, loaders_te['mlp'], criterion, DEVICE)
c_metrics = compute_metrics(c_labels, c_probs)

print(f"\n   Custom MLP Results:")
print(f"   AUC: {c_metrics['AUC-ROC']:.3f} | F1: {c_metrics['F1']:.3f} | CSI: {c_metrics['CSI']:.3f}")
print(f"   Compare with original MLP:")
print(f"   AUC: {test_results['MLP']['AUC-ROC']:.3f} | F1: {test_results['MLP']['F1']:.3f} | CSI: {test_results['MLP']['CSI']:.3f}")

In [ ]:
# ============================================================
#  🧪 EXERCISE 4: Discussion Questions
# ============================================================

questions = """
╔══════════════════════════════════════════════════════════════════╗
║  🌧️  DISCUSSION QUESTIONS for PAGASA Participants               ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. ACCURACY vs CSI                                              ║
║     Why is accuracy misleading for rare events like typhoons?    ║
║     Which metric do you think PAGASA should optimize for?        ║
║                                                                  ║
║  2. POD vs FAR TRADEOFF                                          ║
║     If we lower the probability threshold (e.g., 0.3 instead     ║
║     of 0.5), what happens to POD and FAR?                        ║
║     When would you prefer high POD over low FAR?                 ║
║                                                                  ║
║  3. SPATIAL RESOLUTION                                           ║
║     We used 5.625° resolution (~600 km grid). How would results  ║
║     change with 1.40625° resolution (~150 km)?                   ║
║     What about using station data instead?                       ║
║                                                                  ║
║  4. TYPHOON CASES                                                ║
║     Which model do you think handles typhoon-related heavy rain  ║
║     best, and why? (Hint: think about spatial vs temporal)       ║
║                                                                  ║
║  5. OPERATIONAL USE                                              ║
║     What additional data would you want to add to improve        ║
║     predictions? (e.g., sea surface temperature, CAPE, CIN...)   ║
║                                                                  ║
║  6. LIMITATIONS                                                  ║
║     This model uses only large-scale reanalysis data. What       ║
║     local factors in the Philippines does it miss?               ║
║     (Hint: orography, land-sea breeze, urban heat island...)     ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(questions)

---
## 📝 Step 10: Summary & Next Steps

### What we learned today

| Concept | Where we saw it |
|---------|----------------|
| Reanalysis data (ERA5) | Step 2 — WeatherBench loading |
| Binary classification | Step 4 — heavy rain labels |
| Class imbalance | Step 3 EDA + pos_weight in training |
| Data normalisation | Step 4 — StandardScaler (fit on train only!) |
| MLP | Step 5 — fully connected layers |
| CNN | Step 5 — spatial weather pattern recognition |
| LSTM | Step 5 — temporal sequence modelling |
| Transformer | Step 5 — self-attention across time |
| Backpropagation | Step 6 — how neural networks learn |
| Early stopping | Step 6 — preventing overfitting |
| CSI / HSS / POD / FAR | Step 7 — weather-specific evaluation |
| Probability calibration | Step 8 — do the probabilities mean what they say? |

### 🚀 What to try next

1. **More data:** Use 1.40625° resolution (finer grid, more spatial detail)
2. **More variables:** Add CAPE, SST, outgoing longwave radiation
3. **Spatial-temporal model:** ConvLSTM or 3D-CNN combines spatial + temporal
4. **Station data:** Downscale predictions to individual PAGASA stations
5. **Ensemble:** Average predictions from multiple models for better calibration
6. **Compare with NWP:** How does your model compare to ECMWF or GFS?

### 📚 Key References
- WeatherBench: Rasp et al. (2020), *arXiv:2002.00469*
- WeatherBench2: Rasp et al. (2024), *BAMS*
- Pangu-Weather: Bi et al. (2023), *Nature*
- GraphCast: Lam et al. (2023), *Science*
- Attention Is All You Need: Vaswani et al. (2017), *NeurIPS*

In [ ]:
# ============================================================
#  📝 Final Summary
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════════════╗
║  🎉 WORKSHOP SUMMARY — Philippines Heavy Rain Prediction            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  What we did:                                                        ║
║  ✅ Loaded WeatherBench1 5.625° reanalysis data                      ║
║  ✅ Extracted Philippines region (5°–22°N, 115°–130°E)               ║
║  ✅ Created binary labels: heavy rain = TP ≥ 20 mm/6h                ║
║  ✅ Handled class imbalance with pos_weight                          ║
║  ✅ Trained and compared 4 models:                                   ║
║     🔵 MLP  — flat features, simple baseline                         ║
║     🟢 CNN  — spatial weather maps                                   ║
║     🟡 LSTM — temporal sequences (48h look-back)                     ║
║     🔴 Transformer — self-attention across time                      ║
║  ✅ Evaluated with weather metrics: CSI, HSS, POD, FAR               ║
║  ✅ Checked probability calibration                                  ║
║  ✅ Identified most important predictors                              ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  🚀 NEXT STEPS for PAGASA:                                           ║
║                                                                      ║
║  1. Use REAL WeatherBench1 data (GCS) with longer training period    ║
║  2. Try higher resolution: 1.40625° (64×128 global grid)            ║
║  3. Add more variables: CAPE, SST, outgoing longwave radiation       ║
║  4. Spatial-temporal model: ConvLSTM or 3D CNN                      ║
║  5. Regional downscaling using PAGASA station data                   ║
║  6. Ensemble multiple models for better calibration                  ║
║  7. Evaluate specifically on typhoon vs non-typhoon periods          ║
║  8. Compare with NWP model output (ECMWF, GFS)                      ║
║                                                                      ║
║  📚 REFERENCES:                                                      ║
║  • WeatherBench: Rasp et al. (2020), arXiv:2002.00469               ║
║  • Pangu-Weather: Bi et al. (2023), Nature                          ║
║  • GraphCast: Lam et al. (2023), Science                            ║
║  • WeatherBench2: Rasp et al. (2024), BAMS                          ║
║                                                                      ║
║  💌 Questions? Contact your workshop instructor!                     ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

# Save final results
df_results_clean = df_results.round(3)
df_results_clean.to_csv('model_comparison_results.csv')
print("📄 Results saved to model_comparison_results.csv")
print("\n🇵🇭 Thank you for participating! Mabuhay ang PAGASA! 🌧️")

---
## 🗺️ Spatial Prediction Visualisation

### What exactly are we predicting?

Our models output a **single number per timestep**:
> "Is there heavy rain *anywhere* in the 5×5 Philippines grid?"

```
5×5 grid (25 pixels)           Model output
┌──┬──┬──┬──┬──┐
│  │  │🌧│  │  │
│  │🌧│🌧│  │  │   →   pixel-max > threshold?   →   P(heavy rain) = 0.87
│  │  │🌧│  │  │                                       ↓
│  │  │  │  │  │                               Binary label: 1 = YES 🌧️
│  │  │  │  │  │
└──┴──┴──┴──┴──┘
```

**What the visualisation below shows:**

For a selected test timestep where at least one model predicts heavy rain:
1. **Actual precipitation map** — the true 5×5 rainfall field from WeatherBench
2. **Per-pixel probability proxy** — how much each pixel contributed
   to the heavy rain prediction (based on normalised precipitation)
3. **Binary prediction** — YES 🌧️ / NO ☀️ for each model
4. **Predicted probability** — the raw sigmoid output (0–1) from each model

> ⚠️ **Important limitation:** Our models output *one probability for the whole region*.
> The per-pixel map below shows the **actual observed precipitation**,
> not a per-pixel prediction. True spatial prediction would require
> a separate output for each of the 25 grid points (multi-label or regression).


In [ ]:
# ============================================================
#  🗺️ Spatial Prediction Visualisation (with Philippines map)
# ============================================================

import cartopy.crs as ccrs
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

print("Collecting predictions from all models...")
all_model_results = {}

for model_name, model in models.items():
    model.load_state_dict(all_best_states[model_name])
    model = model.to(DEVICE)
    model.eval()

    loader_type = LOADER_MAP[model_name]
    all_X_list, all_y_list = [], []
    for xb, yb in loaders_te[loader_type]:
        all_X_list.append(xb)
        all_y_list.append(yb)
    all_X_cat = torch.cat(all_X_list, dim=0)
    all_y_cat = torch.cat(all_y_list, dim=0)

    with torch.no_grad():
        probs = torch.sigmoid(model(all_X_cat.to(DEVICE))).cpu().numpy()

    thr_m = test_results[model_name]['Threshold']
    all_model_results[model_name] = {
        'probs':  probs,
        'preds':  (probs >= thr_m).astype(int),
        'labels': all_y_cat.numpy(),
        'Threshold': thr_m,
    }
    print(f"  {model_name:12s} — {len(probs)} test samples")

# --- Find best timestep (true positive with highest probability) ---
ref_name   = 'MLP'
ref_preds  = all_model_results[ref_name]['preds']
ref_labels = all_model_results[ref_name]['labels']

tp_idx = np.where((ref_preds == 1) & (ref_labels == 1))[0]
candidates = tp_idx if len(tp_idx) > 0 else np.where(ref_preds == 1)[0]
best_idx   = candidates[np.argmax(all_model_results[ref_name]['probs'][candidates])]

te_indices_global = np.where(test_mask)[0]
global_idx        = te_indices_global[best_idx]
actual_time       = pd.Timestamp(times_all[global_idx])
tp_time_idx       = global_idx + 1
tp_slice = ph_data['tp'].sel(time=slice(YEAR_START, YEAR_END)).isel(time=tp_time_idx).values

tp_da    = ph_data['tp'].sel(time=slice(YEAR_START, YEAR_END))
lon_vals = tp_da.lon.values   # matches tp_slice shape
lat_vals = tp_da.lat.values   # matches tp_slice shape
proj     = ccrs.PlateCarree()
thr      = test_results[ref_name]['Threshold']

print(f"\nSelected timestep : {actual_time.strftime('%Y-%m-%d %H:%MZ')} → +6h")
print(f"Pixel-max precip  : {float(tp_slice.max()):.1f} mm/6h  (threshold: {thr:.1f} mm/6h)")
print(f"Actual label      : {'🌧️ HEAVY RAIN' if ref_labels[best_idx]==1 else '☀️ No Rain'}")

# ── FIGURE ─────────────────────────────────────────────────────
MODEL_COLORS = {'MLP':'#4472C4','CNN':'#70AD47',
                'LSTM':'#FFC000','Transformer':'#FF4444'}
MODEL_NAMES  = list(models.keys())

fig = plt.figure(figsize=(22, 12))
fig.suptitle(
    f"Spatial Prediction Visualisation\n"
    f"Input: {actual_time.strftime('%Y-%m-%d %H:%MZ')}  →  Target: +6h  |  "
    f"Actual: {'🌧️ HEAVY RAIN' if ref_labels[best_idx]==1 else '☀️ No Rain'}  |  "
    f"Pixel-max: {float(tp_slice.max()):.1f} mm/6h  (threshold: {thr:.1f} mm/6h)",
    fontsize=12, fontweight='bold'
)

outer  = gridspec.GridSpec(2, 1, figure=fig, hspace=0.5, height_ratios=[1.4, 1])
top_gs = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=outer[0], wspace=0.42)

# ── Panel 1: Actual precip on Philippines map ──────────────────
ax_map = fig.add_subplot(top_gs[0], projection=proj)
_, ax_map = make_ph_map(ax=ax_map,
                         title='Actual Precipitation\n(5×5 WeatherBench grid)')

vmax = max(float(tp_slice.max()), thr * 1.5, 1.0)
im   = plot_grid_on_map(tp_slice, lon_vals, lat_vals, ax_map,
                         cmap='Blues', vmin=0, vmax=vmax, alpha=0.8)
plt.colorbar(im, ax=ax_map, shrink=0.75, pad=0.05)

# Annotate each grid cell
for r in range(tp_slice.shape[0]):
    for c in range(tp_slice.shape[1]):
        lat, lon = lat_vals[r], lon_vals[c]
        val    = float(tp_slice[r, c])
        exceed = val >= thr
        ax_map.scatter(lon, lat,
                       s=180 if exceed else 40,
                       c='red' if exceed else 'navy',
                       marker='*' if exceed else 'o',
                       transform=proj, zorder=6)
        ax_map.text(lon + 0.5, lat + 0.6, f'{val:.1f}',
                    fontsize=6.5, color='darkred' if exceed else 'navy',
                    fontweight='bold' if exceed else 'normal',
                    transform=proj, zorder=7)

legend_els = [
    Line2D([0],[0], marker='*', color='w', markerfacecolor='red',
           markersize=11, label=f'≥ {thr:.0f} mm (heavy rain)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='navy',
           markersize=7),
]
ax_map.legend(handles=legend_els, fontsize=7.5, loc='lower left',
              framealpha=0.85)

# ── Panel 2: Per-pixel bar chart ──────────────────────────────
ax_bar = fig.add_subplot(top_gs[1])
pixel_vals = tp_slice.flatten()
pixel_labels = [f'{lat_vals[r]:.0f}°N, {lon_vals[c]:.0f}°E'
                for r in range(N_LAT) for c in range(N_LON)]
bar_colors = ['tomato' if v >= thr else 'steelblue' for v in pixel_vals]
ax_bar.barh(range(len(pixel_vals)), pixel_vals,
            color=bar_colors, edgecolor='white', height=0.75)
ax_bar.axvline(thr, color='red', linestyle='--', linewidth=1.8,
               label=f'Threshold {thr:.1f} mm')
ax_bar.set_yticks(range(len(pixel_vals)))
ax_bar.set_yticklabels(pixel_labels, fontsize=6.5)
ax_bar.set_xlabel('Precipitation (mm/6h)', fontsize=9)
ax_bar.set_title('Per-Pixel Precipitation\n🔴 = exceeds threshold', fontsize=10)
ax_bar.legend(fontsize=8)
ax_bar.grid(axis='x', alpha=0.3)

# ── Panel 3: Explanation box ───────────────────────────────────
ax_txt = fig.add_subplot(top_gs[2])
ax_txt.axis('off')
txt = (
    "What our models predict:\n\n"
    "  Input : 5 atmospheric variables\n"
    "  (z500, q850, t850, u850, v850)\n"
    "  over the 5×5 Philippines grid\n\n"
    "  Output: ONE probability for\n"
    "  the entire region\n\n"
    f"  Label = 1  if  pixel-max\n"
    f"  ≥ {thr:.1f} mm/6h\n\n"
    "  ⚠️ Models do NOT predict\n"
    "  rainfall at each pixel.\n"
    "  The map (left) shows the\n"
    "  actual observed field."
)
ax_txt.text(0.05, 0.95, txt, transform=ax_txt.transAxes,
            fontsize=9.5, verticalalignment='top',
            family='monospace',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

# ── Bottom row: model predictions ─────────────────────────────
bot_gs = gridspec.GridSpecFromSubplotSpec(1, 4, subplot_spec=outer[1], wspace=0.38)

for col, model_name in enumerate(MODEL_NAMES):
    ax = fig.add_subplot(bot_gs[col])
    res   = all_model_results[model_name]
    prob  = float(res['probs'][best_idx])
    pred  = int(res['preds'][best_idx])
    label = int(res['labels'][best_idx])
    thr_m = res['Threshold']
    color = MODEL_COLORS[model_name]

    # Probability gauge bar
    ax.barh([0], [prob],  color=color,       height=0.45, alpha=0.88)
    ax.barh([0], [1.0],   color='lightgrey', height=0.45, alpha=0.25)
    ax.axvline(thr_m, color='red', linestyle='--', linewidth=1.8,
               label=f'Thr {thr_m:.2f}')
    ax.text(min(prob + 0.03, 0.92), 0, f'{prob:.3f}',
            va='center', fontsize=9.5, color=color, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, 0.5)
    ax.set_xlabel('Predicted Probability', fontsize=9)
    ax.set_yticks([])

    is_correct = (pred == label)
    verdict    = '✅ Correct' if is_correct else '❌ Wrong'
    pred_str   = '🌧️ HEAVY'   if pred  == 1 else '☀️ No Rain'
    true_str   = '🌧️ HEAVY'   if label == 1 else '☀️ No Rain'

    info = (f"Probability : {prob:.3f}\n"
            f"Threshold   : {thr_m:.2f}\n"
            f"Prediction  : {pred_str}\n"
            f"Actual      : {true_str}\n"
            f"Result      : {verdict}")
    ax.text(0.5, -0.38, info, transform=ax.transAxes,
            fontsize=8.5, ha='center', va='top',
            bbox=dict(boxstyle='round', alpha=0.12,
                      facecolor='green' if is_correct else 'red'))

    ax.set_title(model_name, fontsize=11, fontweight='bold', color=color)
    ax.legend(fontsize=8, loc='upper right')

plt.savefig('spatial_prediction_vis.png', dpi=130, bbox_inches='tight')
plt.show()
print("\n✅ Spatial prediction visualisation complete!")


In [ ]:
# ============================================================
#  🎁 BONUS: Interactive Prediction Demo
# ============================================================

print("🎯 BONUS: Real-Time Prediction Demo")
print("   Using the best-performing model on the test set\n")

best_name = max(test_results, key=lambda k: test_results[k]['AUC-ROC'])
best_model_final = models[best_name].to(DEVICE)
best_model_final.load_state_dict(all_best_states[best_name])
best_model_final.eval()

threshold_demo = test_results[best_name]['Threshold']
loader_type    = LOADER_MAP[best_name]

# --- Collect ALL test predictions first ---
all_X, all_y = [], []
for xb, yb in loaders_te[loader_type]:
    all_X.append(xb)
    all_y.append(yb)
all_X = torch.cat(all_X, dim=0)
all_y = torch.cat(all_y, dim=0)

with torch.no_grad():
    all_probs = torch.sigmoid(best_model_final(all_X.to(DEVICE))).cpu()

# --- Stratified sampling: guarantee heavy rain cases in demo ---
N_SHOW   = 15
heavy_idx = (all_y == 1).nonzero(as_tuple=True)[0]
clear_idx = (all_y == 0).nonzero(as_tuple=True)[0]

n_heavy = min(5, len(heavy_idx))   # show up to 5 heavy rain cases
n_clear = N_SHOW - n_heavy

# Random sample from each class
g = torch.Generator().manual_seed(42)
sel_heavy = heavy_idx[torch.randperm(len(heavy_idx), generator=g)[:n_heavy]]
sel_clear = clear_idx[torch.randperm(len(clear_idx), generator=g)[:n_clear]]

# Interleave: heavy rain cases spread through the table
indices = torch.cat([sel_heavy, sel_clear])
indices = indices[torch.randperm(len(indices), generator=g)]  # shuffle together

print(f"  Showing {n_heavy} heavy rain + {n_clear} no-rain samples (randomly selected)")
print(f"  Test set: {len(heavy_idx)} heavy rain / {len(all_y)} total ({len(heavy_idx)/len(all_y)*100:.1f}%)\n")
print(f"  {'#':>3} | {'True Label':>12} | {'Predicted Prob':>14} | {'Prediction':>12} | {'Correct?':>8}")
print(f"  {'-'*63}")

for rank, idx in enumerate(indices.tolist(), 1):
    true_label = int(all_y[idx].item())
    prob       = all_probs[idx].item()
    pred       = int(prob >= threshold_demo)
    correct    = '✅' if pred == true_label else '❌'
    true_str   = '🌧️ HEAVY' if true_label == 1 else '☀️ No rain'
    pred_str   = '🌧️ HEAVY' if pred == 1 else '☀️ No rain'
    print(f"  {rank:>3} | {true_str:>12} | {prob:>14.3f} | {pred_str:>12} | {correct:>8}")

print(f"\n   Model: {best_name} | Decision threshold: {threshold_demo:.2f}")
print(f"   Probability > {threshold_demo:.2f} → Heavy Rain Warning Issued 🌧️")
